# Panduan Menjalankan Notebook UGM DAC (Bahasa Indonesia)

Notebook ini berisi pipeline lengkap: setup lingkungan, preprocessing, training 3 model, ensemble submission, dan meta-layer post-processing.

## Cara Menjalankan (Umum)
1. Jalankan cell secara berurutan dari atas ke bawah.
2. Dua cell kode pertama memasang paket utama.
3. Jika memakai Google Colab, jalankan cell mount Google Drive.
4. Lanjutkan cell setup, preprocessing, model, training, dan inferensi sesuai kebutuhan eksperimen.
5. Gunakan cell hotfix atau resume hanya saat training sebelumnya berhenti di tengah jalan.
6. Jalankan bagian meta-layer hanya setelah file probabilitas model sudah tersedia.

## Cara Menjalankan di Kaggle (Khusus)
1. Attach dataset dan bobot model melalui menu Add data di Kaggle Notebook.
2. Ganti semua path Colab atau Drive menjadi path Kaggle.
3. Lewati cell mount Google Drive karena Kaggle tidak memakai drive.mount.
4. Simpan output checkpoint dan submission ke folder tulis Kaggle, misalnya /kaggle/working.
5. Jika resource terbatas, turunkan batch size dan NUM_WORKERS pada cell konfigurasi.

## Path dan Variabel yang Umumnya Perlu Diubah
- GDRIVE: basis path penyimpanan.
- FARIS: root project atau folder kerja utama.
- TRAIN_DIR: folder data train.
- TEST_DIR: folder data test.
- CKPT_DIR: folder simpan checkpoint dan submission.
- DINOV3_VITH_PT: path bobot DINOv3 ViT-H.
- DINOV3_CNX_PT: path bobot DINOv3 ConvNeXt.
- FSFM_CKPT: path bobot FSFM.
- FSFM_MEAN_STD_TXT: path mean-std FSFM.
- YOLO_FACE_PT: path bobot detektor wajah.

## Contoh Adaptasi Path untuk Kaggle
- Sumber dari /content/drive/MyDrive/... ubah ke /kaggle/input/nama-dataset/...
- Output eksperimen simpan ke /kaggle/working
- Jika dataset zip dipakai, pastikan DATASET_ZIP menunjuk file zip di /kaggle/input/...
- Jika dataset sudah diekstrak, pastikan TRAIN_DIR langsung menunjuk folder kelas.

## Penjelasan Cell Kode Pertama
Cell ini memasang dependensi inti (NumPy, PyTorch, torchvision, timm, transformers, dan scikit-learn) agar versi library konsisten sebelum pipeline dijalankan.

In [ ]:
!pip install numpy==2.4.4 torch==2.11.0 torchvision==0.26.0 timm==1.0.20 transformers==4.56.0 scikit-learn==1.8.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 162.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42

## Penjelasan Cell Kode Kedua

Cell ini memasang paket tambahan untuk deteksi wajah dan metrik evaluasi, yaitu ultralytics dan torchmetrics. Pastikan versi paket konsisten agar perilaku inferensi dan training stabil.

In [ ]:
!pip install ultralytics==8.4.33 torchmetrics==1.9.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.6 MB/s eta 0:00:00


# Inisialisasi Lingkungan

Bagian ini menyiapkan koneksi penyimpanan, validasi runtime, cloning repository model, serta preloading detektor wajah agar proses preprocessing multi-thread lebih aman dan cepat.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Penjelasan Cell Mount Drive

Cell ini khusus Google Colab untuk me-mount Google Drive. Jika menjalankan di Kaggle atau lokal, cell ini boleh dilewati selama semua path sudah diarahkan ke folder yang benar.

In [ ]:
"""CELL 1 — Install + Mount"""
import os
import shutil
import subprocess
import sys
from contextlib import nullcontext
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def _pick_existing_path(*candidates: Path, expect_dir: bool = False) -> Path:
    for candidate in candidates:
        p = Path(candidate)
        if p.exists() and (p.is_dir() if expect_dir else p.is_file()):
            return p
    return Path(candidates[0])


def _require_runtime() -> None:
    """
    Validate the environment without mutating numpy or torch in-place.
    Reinstalling NumPy inside an active kernel is what broke the original
    notebook on Colab via an ABI mismatch in numpy.random.
    """
    required = {
        "numpy": "2.4.4",
        "torch": "2.11.0",
        "torchvision": "0.26.0",
        "timm": "1.0.20",
        "transformers": "4.56.0",
        "albumentations": "2.0.8",
        "ultralytics": "8.4.33",
        "scikit-learn": "1.8.0",
        "torchmetrics": "1.9.0",
    }
    missing = []
    for pkg, want in required.items():
        try:
            have = version(pkg)
        except PackageNotFoundError:
            missing.append(f"{pkg}=={want}")
            continue
        print(f"{pkg:15s} {have}")
    if missing:
        raise RuntimeError(
            "Missing runtime packages in the active kernel: "
            + ", ".join(missing)
            + "
Use the local '.venv-fsfm' kernel or install the same packages there."
        )


def _ensure_repo(url: str, repo_root: Path, marker: str) -> None:
    if (repo_root / marker).exists():
        return
    if repo_root.exists():
        shutil.rmtree(repo_root)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo_root)])


def _resolve_dataset_extract_dir() -> Path:
    colab_dir = Path("/content/dataset_rotated")
    return colab_dir if colab_dir.exists() else (PROJECT_ROOT / "dataset_rotated")


def _resolve_dataset_zip() -> Path:
    return _pick_existing_path(
        Path("/content/drive/MyDrive/faris/dataset_rotated.zip"),
        PROJECT_ROOT / "dataset_rotated.zip",
    )


_require_runtime()

FSFM_REPO = "/tmp/fsfm"
FSFM_DIR  = "/tmp/fsfm/fsfm-3c"   # ← models_vit.py lives HERE, not repo root

if not os.path.exists(f"{FSFM_DIR}/models_vit.py"):
    _ensure_repo("https://github.com/wolo-wolo/FSFM-CVPR25.git", Path(FSFM_REPO), "fsfm-3c/models_vit.py")
    print(f"FSFM cloned ✓  models_vit.py exists: {os.path.exists(f'{FSFM_DIR}/models_vit.py')}")
else:
    print("FSFM already cloned ✓")

# Clone DINOv3
DINOV3_DIR = "/tmp/dinov3"
if not os.path.exists(f"{DINOV3_DIR}/hubconf.py"):
    _ensure_repo("https://github.com/facebookresearch/dinov3.git", Path(DINOV3_DIR), "hubconf.py")
    print(f"DINOv3 cloned ✓")
else:
    print("DINOv3 already cloned ✓")

# Pre-download YOLO in main thread BEFORE workers spawn (prevents 100x parallel download)
import threading
_face_lock       = threading.Lock()
_face_detector   = None

def get_face_detector():
    global _face_detector
    if _face_detector is not None:
        return _face_detector
    with _face_lock:
        if _face_detector is not None:
            return _face_detector
        from ultralytics import YOLO
        drive_yolo_pt = Path("/content/drive/MyDrive/data_dac/weights/yolov11n-face.pt")
        local_yolo_pt = PROJECT_ROOT / "weights" / "yolov11n-face.pt"
        yolo_pt = drive_yolo_pt if drive_yolo_pt.exists() else local_yolo_pt
        if not yolo_pt.exists():
            print("Downloading yolov11n-face.pt ...")
            import urllib.request
            yolo_pt.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/YapaLab/yolo-face/releases/download/1.0.0/yolov11n-face.pt",
                str(yolo_pt),
            )
        _face_detector = YOLO(str(yolo_pt))
    return _face_detector
print("Pre-loading face detector in main thread ...")
get_face_detector()
print("Face detector ready ✓")

import torch
print(f"PyTorch:  {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("GPU:      CPU-only runtime")


numpy           2.4.4
torch           2.11.0
torchvision     0.26.0
timm            1.0.20
transformers    4.56.0
albumentations  2.0.8
ultralytics     8.4.33
scikit-learn    1.8.0
torchmetrics    1.9.0
FSFM cloned ✓  models_vit.py exists: True
DINOv3 cloned ✓
Pre-loading face detector in main thread ...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Face detector ready ✓
PyTorch:  2.11.0+cu130
GPU:      NVIDIA L4
VRAM:     23.7 GB


## Penjelasan Cell Setup Runtime

Cell ini adalah setup inti: validasi dependensi aktif, cloning repository FSFM dan DINOv3 bila belum ada, penentuan path dataset, pre-download model YOLO wajah, serta pengecekan perangkat komputasi (GPU atau CPU).

In [ ]:
import zipfile

DATASET_ZIP = _resolve_dataset_zip()
DATASET_EXTRACT_DIR = _resolve_dataset_extract_dir()

if DATASET_EXTRACT_DIR.exists():
    print(f"Dataset already extracted: {DATASET_EXTRACT_DIR}")
elif DATASET_ZIP.exists():
    print(f"Extracting {DATASET_ZIP} → {DATASET_EXTRACT_DIR}")
    DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(DATASET_EXTRACT_DIR)
else:
    raise FileNotFoundError(
        f"dataset_rotated.zip not found at either {Path('/content/drive/MyDrive/faris/dataset_rotated.zip')} "
        f"or {PROJECT_ROOT / 'dataset_rotated.zip'}"
    )


Extracting /content/drive/MyDrive/faris/dataset_printed_mask_fix.zip → /content/dataset_printed_mask_fix


## Penjelasan Cell Ekstraksi Dataset

Cell ini mengecek apakah dataset sudah diekstrak. Jika belum, cell akan mengekstrak file zip ke folder tujuan. Jika file zip tidak ditemukan, proses dihentikan dengan error agar path segera diperbaiki.

# Konfigurasi Path dan Hyperparameter

Bagian ini mendefinisikan seluruh path data, path checkpoint, konfigurasi device, seed, jumlah worker, serta konfigurasi model yang dipakai selama training dan inferensi.

In [ ]:
"""CELL 2 — Configuration"""
import os, random, math
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
import numpy as np
import torch

# ── Paths ─────────────────────────────────────────────────────────────────────
GDRIVE         = _pick_existing_path(Path("/content/drive/MyDrive"), PROJECT_ROOT, expect_dir=True)
FARIS          = _pick_existing_path(GDRIVE / "faris", PROJECT_ROOT, expect_dir=True)

TRAIN_DIR      = DATASET_EXTRACT_DIR
TEST_DIR       = _pick_existing_path(GDRIVE / "data_dac/datatest/test_images", PROJECT_ROOT / "test_images", expect_dir=True)
CKPT_DIR       = PROJECT_ROOT / "checkpoint" if FARIS == PROJECT_ROOT else FARIS / "checkpoint"

# Augmentation → /dev/shm (RAM tmpfs, fast I/O on Colab L4)
AUG_DIR        = Path("/dev/shm/fas_aug")
CACHE_DIR      = Path("/dev/shm/fas_cache")

# Model weight files
DINOV3_VITH_PT  = _pick_existing_path(
    FARIS / "dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth",
    PROJECT_ROOT / "dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth",
)
DINOV3_CNX_PT   = _pick_existing_path(
    FARIS / "dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth",
    PROJECT_ROOT / "dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth",
)
# FSFM checkpoint — downloaded here if not already present
FSFM_CKPT     = _pick_existing_path(FARIS / "fsfm_vit_b_vf2_400e.pth", PROJECT_ROOT / "fsfm_vit_b_vf2_400e.pth")
FSFM_DIR      = Path("/tmp/fsfm/fsfm-3c")   # ← was Path("/tmp/fsfm")

# FSFM normalization stats — downloaded alongside checkpoint
FSFM_MEAN_STD_TXT = _pick_existing_path(FARIS / "fsfm_pretrain_mean_std.txt", PROJECT_ROOT / "fsfm_pretrain_mean_std.txt")


def ensure_fsfm_assets() -> None:
    from huggingface_hub import hf_hub_download

    if not FSFM_CKPT.exists():
        print("Downloading FSFM checkpoint (~1.6GB) ...")
        _dl = hf_hub_download(
            repo_id="Wolowolo/fsfm-3c",
            filename="pretrained_models/VF2_ViT-B/checkpoint-400.pth",
            local_dir=str(FARIS),
            local_dir_use_symlinks=False,
        )
        shutil.copy(_dl, str(FSFM_CKPT))
        print(f"  Saved to: {FSFM_CKPT}")
    else:
        print(f"FSFM checkpoint: {FSFM_CKPT.name} ✓")

    if not FSFM_MEAN_STD_TXT.exists():
        _dl = hf_hub_download(
            repo_id="Wolowolo/fsfm-3c",
            filename="pretrained_models/VF2_ViT-B/pretrain_ds_mean_std.txt",
            local_dir=str(FARIS),
            local_dir_use_symlinks=False,
        )
        shutil.copy(_dl, str(FSFM_MEAN_STD_TXT))
        print(f"  Saved norm stats to: {FSFM_MEAN_STD_TXT}")
    else:
        print(f"FSFM norm stats:  {FSFM_MEAN_STD_TXT.name} ✓")

# Repo dirs (cloned in cell 1)
FSFM_DIR       = Path("/tmp/fsfm/fsfm-3c")
DINOV3_DIR     = Path("/tmp/dinov3")

# Face detector weights (auto-download via ultralytics)
YOLO_FACE_PT   = (
    Path("/content/drive/MyDrive/data_dac/weights/yolov11n-face.pt")
    if Path("/content/drive/MyDrive/data_dac/weights/yolov11n-face.pt").exists()
    else PROJECT_ROOT / "weights" / "yolov11n-face.pt"
)

for p in [CKPT_DIR, AUG_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ── Classes ────────────────────────────────────────────────────────────────────
CLASS_NAMES: List[str] = [
    "fake_mannequin", "fake_mask", "fake_printed",
    "fake_screen",    "fake_unknown", "realperson",
]
NUM_CLASSES               = len(CLASS_NAMES)
CLASS_TO_IDX              = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS              = {i: c for c, i in CLASS_TO_IDX.items()}
REAL_IDX                  = CLASS_TO_IDX["realperson"]
FAKE_MASK_IDX             = CLASS_TO_IDX["fake_mask"]

# ── Augmentation multipliers ───────────────────────────────────────────────────
AUG_MULT: Dict[str, int] = {
    "fake_printed":   4,
    "fake_screen":    2,
    "fake_mannequin": 2,
    "fake_mask":      1,
    "fake_unknown":   0,
    "realperson":     0,
}

# ── Hardware ───────────────────────────────────────────────────────────────────
CPU_COUNT    = os.cpu_count() or 2
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VRAM_GB      = (torch.cuda.get_device_properties(0).total_memory / 1e9) if DEVICE.type == "cuda" else 0.0
SEED         = 42
AMP_DTYPE    = torch.bfloat16 if DEVICE.type == "cuda" else None
ENABLE_COMPILE = os.environ.get("FSFM_ENABLE_COMPILE", "0") == "1" and DEVICE.type == "cuda" and VRAM_GB >= 12
SMOKE_TRAIN_STEPS = int(os.environ.get("FSFM_MAX_TRAIN_STEPS", "0") or "0")
SMOKE_VAL_STEPS   = int(os.environ.get("FSFM_MAX_VAL_STEPS", "0") or "0")
NUM_WORKERS  = 0 if (SMOKE_TRAIN_STEPS or VRAM_GB < 8) else min(4, max(1, CPU_COUNT // 2))
PREFETCH     = 2
PIN_MEMORY   = DEVICE.type == "cuda"


def _fit_batch(base_batch_size: int) -> int:
    if DEVICE.type != "cuda":
        return 1
    if VRAM_GB >= 20:
        return base_batch_size
    if VRAM_GB >= 12:
        return max(2, base_batch_size // 4)
    return 1


def autocast_ctx():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return nullcontext()

# ── Per-model configs ──────────────────────────────────────────────────────────
@dataclass
class ModelCfg:
    name:                str
    resolution:          int
    batch_size:          int
    backbone_lr:         float
    head_lr:             float
    epochs:              int   = 60
    warmup_ratio:        float = 0.05
    weight_decay:        float = 1e-4
    early_stop_patience: int   = 15
    overfit_delta:       float = 0.015
    lora_r:              int   = 8
    lora_alpha:          int   = 16
    lora_dropout:        float = 0.05
    binary_weight:       float = 0.3
    label_smoothing:     float = 0.1
    use_fft:             bool  = False
    grad_checkpoint:     bool  = True

DINOV3_VITH_CFG = ModelCfg(
    name         = "dinov3_vith",
    resolution   = 448,
    batch_size   = _fit_batch(16),
    backbone_lr  = 5e-6,
    head_lr      = 1e-4,
)

DINOV3_CNX_CFG = ModelCfg(
    name         = "dinov3_cnx",
    resolution   = 384,
    batch_size   = _fit_batch(16),
    backbone_lr  = 5e-6,
    head_lr      = 1e-4,
    use_fft      = True,
    grad_checkpoint = False,  # ConvNeXt doesn't support it same way
)

FSFM_CFG = ModelCfg(
    name         = "fsfm",
    resolution   = 224,
    batch_size   = _fit_batch(32),
    backbone_lr  = 5e-6,
    head_lr      = 1e-4,
)

ALL_CFGS = [DINOV3_VITH_CFG, DINOV3_CNX_CFG, FSFM_CFG]

# ── ImageNet normalization (all three models use it) ──────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ── Val split ─────────────────────────────────────────────────────────────────
VAL_FRACTION = 0.20

# ── Seed ──────────────────────────────────────────────────────────────────────
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print("Config ready.")
print(f"Runtime root: {PROJECT_ROOT}")
print(f"Train dir   : {TRAIN_DIR}")
print(f"Checkpoints : {CKPT_DIR}")
print(f"Device      : {DEVICE}  compile={ENABLE_COMPILE}  workers={NUM_WORKERS}")
for cfg in ALL_CFGS:
    print(f"  {cfg.name:16s} res={cfg.resolution} bs={cfg.batch_size} "
          f"blr={cfg.backbone_lr} fft={cfg.use_fft}")


Config ready.
Runtime root: /content
Train dir   : /content/dataset_printed_mask_fix
Checkpoints : /content/drive/MyDrive/faris/checkpoint
Device      : cuda  compile=False  workers=4
  dinov3_vith      res=448 bs=16 blr=5e-06 fft=False
  dinov3_cnx       res=384 bs=16 blr=5e-06 fft=True
  fsfm             res=224 bs=32 blr=5e-06 fft=False


## Penjelasan Cell Konfigurasi Global

Cell ini menetapkan variabel konfigurasi utama, termasuk struktur kelas, aturan augmentasi, skala batch berdasarkan VRAM, serta konfigurasi tiga model (DINOv3 ViT-H, DINOv3 ConvNeXt, dan FSFM). Jika Anda pindah environment, cell ini adalah tempat utama untuk menyesuaikan path dan parameter.

In [ ]:
"""CELL 3 — Preprocessing
Fixes applied:
  #3  — YOLO receives BGR numpy (Ultralytics numpy API expects BGR)
  #9  — augmented crops use face detection, not just center crop
  #13 — FFT transform pipeline split: spatial → FFT on [0,1] → normalize RGB
  #14 — /dev/shm space check with fallback to /tmp
  #15 — fixed done counter in preprocess_all
  #16 — cv2.imwrite return value checked
"""

import cv2
import os
import shutil
import concurrent.futures
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image, ImageOps
import torch

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# ── /dev/shm space check (#14) ─────────────────────────────────────────────────

def _get_cache_root() -> Path:
    """Use /dev/shm (RAM) if ≥4GB available, else fallback to /tmp."""
    shm = Path("/dev/shm")
    try:
        stat = shutil.disk_usage(str(shm))
        if stat.free >= 4 * 1024**3:
            return shm
    except Exception:
        pass
    print("[WARN] /dev/shm has < 4GB free — using /tmp instead.")
    return Path("/tmp")

_CACHE_ROOT = _get_cache_root()
AUG_DIR   = _CACHE_ROOT / "fas_aug"
CACHE_DIR = _CACHE_ROOT / "fas_cache"

for p in [AUG_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Cache root: {_CACHE_ROOT}  (free: {shutil.disk_usage(str(_CACHE_ROOT)).free/1e9:.1f} GB)")

# ── Face detector ──────────────────────────────────────────────────────────────
_face_detector = None
_face_infer_lock = threading.Lock()

def get_face_detector():
    global _face_detector
    if _face_detector is not None:
        return _face_detector
    from ultralytics import YOLO
    if YOLO_FACE_PT.exists():
        _face_detector = YOLO(str(YOLO_FACE_PT))
    else:
        print("Downloading yolov11n-face.pt ...")
        import urllib.request
        url = "https://github.com/YapaLab/yolo-face/releases/download/1.0.0/yolov11n-face.pt"
        YOLO_FACE_PT.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, str(YOLO_FACE_PT))
        _face_detector = YOLO(str(YOLO_FACE_PT))
    return _face_detector


# ── Image loading (EXIF-aware) ─────────────────────────────────────────────────

def load_image(path: Path) -> np.ndarray:
    """Load RGB uint8 image with EXIF orientation applied."""
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        if img.mode != "RGB":
            img = img.convert("RGB")
        return np.array(img)


def save_image(img: np.ndarray, path: Path, quality: int = 95) -> bool:
    """Save RGB image. Returns True on success. (#16: check return value)"""
    path.parent.mkdir(parents=True, exist_ok=True)
    bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    ok  = cv2.imwrite(str(path), bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        print(f"  [WARN] cv2.imwrite failed: {path}")
    return ok


# ── Face crop (#3: BGR for YOLO) ────────────────────────────────────────────────

def detect_face_crop(img_rgb: np.ndarray, margin: float = 0.20) -> np.ndarray:
    """
    Detect face with yolov12n-face.
    IMPORTANT: Ultralytics expects BGR numpy arrays (#3).
    Falls back to 80% center crop on failure.
    """
    detector = get_face_detector()
    H, W = img_rgb.shape[:2]
    try:
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)  # fix #3
        with _face_infer_lock:
            results  = detector(img_bgr, verbose=False, conf=0.40, device="cpu")
        boxes    = results[0].boxes
        if boxes is not None and len(boxes) > 0:
            best       = int(boxes.conf.argmax())
            x1,y1,x2,y2 = boxes.xyxy[best].cpu().numpy().astype(int)
            bw, bh     = max(1, x2-x1), max(1, y2-y1)
            mx, my     = int(bw*margin), int(bh*margin)
            x1 = max(0, x1-mx); y1 = max(0, y1-my)
            x2 = min(W, x2+mx); y2 = min(H, y2+my)
            if x2 > x1 and y2 > y1:
                return img_rgb[y1:y2, x1:x2]
    except Exception as e:
        pass
    # Fallback: 80% center crop
    s = int(min(H, W) * 0.80)
    cy, cx = H//2, W//2
    y1 = max(0, cy - s//2); y2 = min(H, y1 + s)
    x1 = max(0, cx - s//2); x2 = min(W, x1 + s)
    return img_rgb[y1:y2, x1:x2]


# ── Preprocessing ──────────────────────────────────────────────────────────────

def _preprocess_one(args: Tuple) -> Tuple[bool, str]:
    """Returns (success, src_str)."""
    src_str, full_dst_str, crop_dst_str, resolution = args
    full_dst = Path(full_dst_str)
    crop_dst = Path(crop_dst_str)
    if full_dst.exists() and crop_dst.exists():
        return True, src_str
    try:
        img = load_image(Path(src_str))
        full = cv2.resize(img, (resolution, resolution),
                         interpolation=cv2.INTER_LANCZOS4)
        if not save_image(full, full_dst):
            return False, src_str
        if not crop_dst.exists():
            crop = detect_face_crop(img)
            crop = cv2.resize(crop, (resolution, resolution),
                             interpolation=cv2.INTER_LANCZOS4)
            if not save_image(crop, crop_dst):
                return False, src_str
        return True, src_str
    except Exception as e:
        print(f"  [WARN] {Path(src_str).name}: {e}")
        return False, src_str


def preprocess_all(
    samples: List[Tuple[Path, int]],
    cache_dir: Path,
    resolution: int,
    max_workers: int = 12,
) -> Dict[str, Tuple[Path, Path]]:
    """
    Returns {str(original_path): (full_cached_path, crop_cached_path)}.
    Keys are ORIGINAL source paths so submission IDs use original stems. (#2)
    """
    (cache_dir / "full").mkdir(parents=True, exist_ok=True)
    (cache_dir / "crop").mkdir(parents=True, exist_ok=True)

    tasks, path_map = [], {}
    for path, label in samples:
        prefix = CLASS_NAMES[label] if label >= 0 else "test"
        # Include parent dir name to avoid stem collisions within same class (#10)
        stem   = f"{prefix}_{path.parent.name}_{path.stem}"
        full_p = cache_dir / "full" / f"{stem}.jpg"
        crop_p = cache_dir / "crop" / f"{stem}.jpg"
        path_map[str(path)] = (full_p, crop_p)  # key = original path (#2)
        if not (full_p.exists() and crop_p.exists()):
            tasks.append((str(path), str(full_p), str(crop_p), resolution))

    if tasks:
        print(f"Preprocessing {len(tasks)} → {cache_dir} ({max_workers} threads)")
        futures = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(_preprocess_one, t) for t in tasks]
            results = [f.result() for f in concurrent.futures.as_completed(futures)]
        ok = sum(1 for success, _ in results if success)  # fix #15
        print(f"  Done: {ok}/{len(tasks)}")
    else:
        print(f"Cache: all {len(samples)} present at {cache_dir}")
    return path_map


def build_train_samples(train_dir: Path) -> List[Tuple[Path, int]]:
    samples = []
    for cls in CLASS_NAMES:
        cls_dir = train_dir / cls
        if not cls_dir.exists():
            continue
        for p in sorted(cls_dir.glob("*")):
            if p.suffix.lower() in EXTS:
                samples.append((p, CLASS_TO_IDX[cls]))
    return samples


def build_test_samples(test_dir: Path) -> List[Tuple[Path, int]]:
    return sorted(
        [(p, -1) for p in test_dir.glob("*") if p.suffix.lower() in EXTS],
        key=lambda x: x[0].name,
    )


# ── Offline augmentation (#9: face detection for aug crops) ────────────────────

def _aug_pipeline(cls_name: str) -> A.Compose:
    base = [A.HorizontalFlip(p=0.5)]
    if cls_name == "fake_printed":
        return A.Compose(base + [
            A.ImageCompression(quality_range=(35, 80), p=0.8),
            A.GaussNoise(std_range=(0.008, 0.039), p=0.6),
            A.ElasticTransform(alpha=25, sigma=4, p=0.4),
            A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, p=0.5),
        ])
    elif cls_name == "fake_screen":
        return A.Compose(base + [
            A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.05, p=0.7),
            A.ImageCompression(quality_range=(55, 95), p=0.5),
            A.GaussNoise(std_range=(0.006, 0.024), p=0.5),
        ])
    elif cls_name == "fake_mannequin":
        return A.Compose(base + [
            A.ColorJitter(brightness=0.3, contrast=0.2, saturation=0.2, hue=0.08, p=0.8),
            A.HueSaturationValue(10, 20, 20, p=0.5),
            A.GaussNoise(std_range=(0.004, 0.016), p=0.3),
        ])
    elif cls_name == "fake_mask":
        return A.Compose(base + [
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
            A.Perspective(scale=(0.02, 0.06), p=0.4),
            A.ColorJitter(brightness=0.2, contrast=0.15, p=0.5),
        ])
    return A.Compose(base)


def _aug_one(args: Tuple) -> bool:
    src_str, full_dst_str, crop_dst_str, cls_name, resolution = args
    full_dst = Path(full_dst_str)
    crop_dst = Path(crop_dst_str)
    if full_dst.exists() and crop_dst.exists():
        return True
    try:
        img = load_image(Path(src_str))
        aug = _aug_pipeline(cls_name)
        out = aug(image=img)["image"]

        # Full image
        full_r = cv2.resize(out, (resolution, resolution),
                            interpolation=cv2.INTER_LANCZOS4)
        if not save_image(full_r, full_dst, quality=90):
            return False

        # Crop: face detect on augmented image (#9)
        crop   = detect_face_crop(out)
        crop_r = cv2.resize(crop, (resolution, resolution),
                            interpolation=cv2.INTER_LANCZOS4)
        if not save_image(crop_r, crop_dst, quality=90):
            return False
        return True
    except Exception as e:
        return False


def run_offline_augmentation(
    train_orig:   List[Tuple[Path, int]],
    aug_dir:      Path,
    resolution:   int,
    max_workers:  int = 12,
) -> List[Tuple[Tuple[Path, Path], int]]:
    """
    Source: original (non-cached) images so augmentation applies before resize.
    Returns list of ((full_aug_path, crop_aug_path), label).
    """
    (aug_dir / "full").mkdir(parents=True, exist_ok=True)
    (aug_dir / "crop").mkdir(parents=True, exist_ok=True)

    tasks, results_map = [], {}
    for path, label in train_orig:
        cls  = CLASS_NAMES[label]
        mult = AUG_MULT.get(cls, 0)
        if mult == 0:
            continue
        for i in range(mult):
            stem     = f"aug{i}_{path.parent.name}_{path.stem}"
            full_dst = aug_dir / "full" / f"{stem}.jpg"
            crop_dst = aug_dir / "crop" / f"{stem}.jpg"
            tasks.append((str(path), str(full_dst), str(crop_dst), cls, resolution))
            results_map[(str(full_dst), str(crop_dst))] = label

    if tasks:
        print(f"Augmenting {len(tasks)} → {aug_dir} ({max_workers} threads, RAM)...")
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
            ok = sum(ex.map(_aug_one, tasks))
        print(f"  Done: {ok}/{len(tasks)}")

    return [
        ((Path(fd), Path(cd)), lbl)
        for (fd, cd), lbl in results_map.items()
        if Path(fd).exists() and Path(cd).exists()
    ]


# ── Transform pipelines (#13: FFT computed before normalization) ────────────────
# Strategy: split into spatial_transform (no normalize) + normalize_transform.
# Dataset computes FFT on [0,1] tensor BEFORE normalization, then normalizes RGB.

def get_spatial_transform(resolution: int, augment: bool) -> A.Compose:
    """Spatial transforms only — no normalize."""
    tfms = [A.Resize(resolution, resolution)]
    if augment:
        tfms += [
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=12, p=0.4),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.7),
            A.OneOf([
                A.GaussNoise(std_range=(0.008, 0.031)),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
                A.ImageCompression(quality_range=(50, 95)),
            ], p=0.5),
            A.OneOf([
                A.GridDistortion(num_steps=5, distort_limit=0.3),
                A.OpticalDistortion(distort_limit=0.3),
            ], p=0.3),
            A.CoarseDropout(num_holes_range=(1,6),
                            hole_height_range=(8,32), hole_width_range=(8,32), p=0.3),
        ]
    tfms += [ToTensorV2()]   # HWC→CHW, uint8→float32/255
    return A.Compose(tfms)


# ImageNet normalization for DINOv3 ViT-H+ and ConvNeXt
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
_imagenet_norm = A.Normalize(mean=list(IMAGENET_MEAN), std=list(IMAGENET_STD))

# FSFM normalization — loaded from pretrain_ds_mean_std.txt; overwritten in cell 5
FSFM_MEAN = (0.5, 0.5, 0.5)   # placeholder, cell 5 overwrites this
FSFM_STD  = (0.5, 0.5, 0.5)


def compute_fft_channels(img_01: torch.Tensor) -> torch.Tensor:
    """
    Log-magnitude FFT per channel, normalized [0,1].
    Input: [3, H, W] float32 in [0,1] range — BEFORE ImageNet normalization. (#13)
    """
    out = []
    for c in range(3):
        f   = torch.fft.fftshift(torch.fft.fft2(img_01[c]))
        mag = torch.log1p(torch.abs(f))
        mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
        out.append(mag)
    return torch.stack(out)   # [3, H, W] in [0,1]


def apply_imagenet_norm(img_01: torch.Tensor) -> torch.Tensor:
    """Normalize [0,1] CHW tensor with ImageNet stats."""
    mean = torch.tensor(IMAGENET_MEAN, dtype=img_01.dtype).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  dtype=img_01.dtype).view(3, 1, 1)
    return (img_01 - mean) / std


def apply_fsfm_norm(img_01: torch.Tensor) -> torch.Tensor:
    """Normalize [0,1] CHW tensor with FSFM pretraining stats."""
    mean = torch.tensor(FSFM_MEAN, dtype=img_01.dtype).view(3, 1, 1)
    std  = torch.tensor(FSFM_STD,  dtype=img_01.dtype).view(3, 1, 1)
    return (img_01 - mean) / std


print("Preprocessing ready.")
print(f"Cache: {CACHE_DIR}")
print(f"Aug  : {AUG_DIR}")


Cache root: /dev/shm  (free: 27.5 GB)
Preprocessing ready.
Cache: /dev/shm/fas_cache
Aug  : /dev/shm/fas_aug


## Penjelasan Cell Preprocessing

Cell ini menangani seluruh preprocessing gambar: load image dengan koreksi EXIF, resize, deteksi wajah untuk crop, caching ke RAM atau disk, augmentasi offline, hingga persiapan transform spasial dan FFT. Bagian ini krusial karena kualitas data masuk sangat memengaruhi performa model.

In [ ]:
"""CELL 3b — YOLO race-condition fix
The face detector must be downloaded ONCE in the main thread before
worker threads start. Also adds a threading lock so workers don't
race even if called simultaneously.
"""
import threading
from pathlib import Path

# ── Thread lock to prevent simultaneous downloads ──────────────────────────────
_face_detector_lock = threading.Lock()
_face_infer_lock    = threading.Lock()
_face_detector      = None   # reset global

def get_face_detector():
    """Thread-safe singleton face detector."""
    global _face_detector
    if _face_detector is not None:
        return _face_detector
    with _face_detector_lock:
        if _face_detector is not None:   # double-checked locking
            return _face_detector
        from ultralytics import YOLO
        if YOLO_FACE_PT.exists():
            print(f"  Loading YOLO from Drive: {YOLO_FACE_PT}")
            _face_detector = YOLO(str(YOLO_FACE_PT))
        else:
            print("  Downloading yolov11n-face.pt (once) ...")
            import urllib.request
            url = "https://github.com/YapaLab/yolo-face/releases/download/1.0.0/yolov11n-face.pt"
            YOLO_FACE_PT.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, str(YOLO_FACE_PT))
            print(f"  Saved to {YOLO_FACE_PT} ✓")
            _face_detector = YOLO(str(YOLO_FACE_PT))
    return _face_detector

# ── Pre-warm in main thread BEFORE any workers spawn ──────────────────────────
print("Pre-loading face detector in main thread ...")
get_face_detector()
print("Face detector ready ✓")


Pre-loading face detector in main thread ...
  Loading YOLO from Drive: /content/drive/MyDrive/data_dac/weights/yolov11n-face.pt
Face detector ready ✓


## Penjelasan Cell Hotfix YOLO Threading

Cell ini memperkuat inisialisasi detektor wajah agar thread-safe. Tujuannya mencegah kondisi race saat banyak worker berjalan bersamaan, terutama ketika file bobot YOLO perlu diunduh atau diload sekali di awal proses.

In [ ]:
"""CELL 4 — Dataset
Fixes:
  #8  — worker_init_fn seeds numpy + Python random + torch RNG
  #13 — FFT computed on [0,1] before normalization, using split transform pipeline
"""

import cv2
import random as _random
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import numpy as np
import albumentations as A


class FASDataset(Dataset):
    """
    Returns per-sample dict:
      image_full [C,H,W]  — full image tensor (C=3 or 6 for ConvNeXt)
      image_crop [C,H,W]  — face crop tensor (C=3 or 6 for ConvNeXt)
      label
      path  (original source path stem for submission ID lookup)
    """

    def __init__(
        self,
        samples:        List[Tuple[Tuple[Path, Path], int]],
        spatial_tf:     A.Compose,
        norm_fn,                         # callable: (tensor [3,H,W]) → tensor
        is_train:       bool  = False,
        use_fft:        bool  = False,
        pda_real_full:  Optional[List[Path]] = None,
        pda_prob:       float = 0.4,
        pda_patches:    int   = 3,
        pda_patch_size: int   = 48,
        orig_paths:     Optional[List[str]] = None,   # original stems for IDs
    ):
        self.samples       = samples
        self.spatial_tf    = spatial_tf
        self.norm_fn       = norm_fn
        self.is_train      = is_train
        self.use_fft       = use_fft
        self.pda_real_full = pda_real_full or []
        self.pda_prob      = pda_prob
        self.pda_patches   = pda_patches
        self.pda_sz        = pda_patch_size
        self.orig_paths    = orig_paths or [str(fp) for (fp, _), _ in samples]

    def __len__(self): return len(self.samples)

    def _apply_pda(self, img: np.ndarray) -> np.ndarray:
        if not self.pda_real_full:
            return img
        src_path = self.pda_real_full[np.random.randint(len(self.pda_real_full))]
        try:
            src = load_image(src_path)
            src = cv2.resize(src, (img.shape[1], img.shape[0]),
                             interpolation=cv2.INTER_LINEAR)
        except Exception:
            return img
        H, W = img.shape[:2]; sz = self.pda_sz
        out = img.copy()
        for _ in range(self.pda_patches):
            if H <= sz or W <= sz:
                break
            y = np.random.randint(0, H - sz)
            x = np.random.randint(0, W - sz)
            out[y:y+sz, x:x+sz] = src[y:y+sz, x:x+sz]
        return out

    def _to_tensor(self, img: np.ndarray) -> torch.Tensor:
        """
        Spatial transform → [0,1] tensor → FFT (if needed) → normalize.
        FFT is computed on [0,1] range, before normalization. (#13)
        """
        # spatial_tf ends with ToTensorV2 which gives CHW float32 [0,255]
        t = self.spatial_tf(image=img)["image"].float() / 255.0   # CHW [0,1]

        if self.use_fft:
            fft = compute_fft_channels(t)          # [3, H, W] in [0,1]
            t_n = self.norm_fn(t)                  # normalize RGB
            t   = torch.cat([t_n, fft], dim=0)     # [6, H, W]
        else:
            t = self.norm_fn(t)                    # normalize RGB, [3, H, W]
        return t

    def __getitem__(self, idx: int) -> Dict:
        (full_p, crop_p), label = self.samples[idx]
        orig_stem = Path(self.orig_paths[idx]).stem

        full_img = load_image(Path(full_p))
        crop_img = load_image(Path(crop_p))

        if (self.is_train and label == FAKE_MASK_IDX
                and np.random.random() < self.pda_prob):
            full_img = self._apply_pda(full_img)

        return {
            "image_full": self._to_tensor(full_img),
            "image_crop": self._to_tensor(crop_img),
            "label":      torch.tensor(label, dtype=torch.long),
            "path":       orig_stem,   # original stem for submission ID (#2)
        }


def worker_init_fn(worker_id: int) -> None:
    """Seed numpy + Python random + torch for each worker. (#8)"""
    info   = torch.utils.data.get_worker_info()
    seed   = SEED + worker_id + (info.id if info is not None else 0)
    seed32 = seed % (2**31)
    np.random.seed(seed32)
    _random.seed(seed32)
    torch.manual_seed(seed32)


def make_loader(
    samples:       List[Tuple[Tuple[Path, Path], int]],
    spatial_tf:    A.Compose,
    norm_fn,
    shuffle:       bool,
    batch_size:    int,
    is_train:      bool  = False,
    use_fft:       bool  = False,
    pda_real_full: Optional[List[Path]] = None,
    orig_paths:    Optional[List[str]]  = None,
    drop_last:     bool  = False,
) -> DataLoader:
    ds = FASDataset(
        samples, spatial_tf, norm_fn,
        is_train=is_train, use_fft=use_fft,
        pda_real_full=pda_real_full,
        orig_paths=orig_paths,
    )
    kwargs = {}
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = PREFETCH
        kwargs["persistent_workers"] = True
    return DataLoader(
        ds,
        batch_size         = batch_size,
        shuffle            = shuffle,
        num_workers        = NUM_WORKERS,
        pin_memory         = PIN_MEMORY,
        drop_last          = drop_last,
        worker_init_fn     = worker_init_fn,
        **kwargs,
    )


def mild_class_weights(
    samples: List[Tuple[Tuple[Path, Path], int]],
    device:  torch.device,
) -> torch.Tensor:
    counts = np.bincount([l for _, l in samples], minlength=NUM_CLASSES).astype(float)
    counts = np.maximum(counts, 1.0)
    w = 1.0 / counts; w = w / w.mean()
    print("Class weights (post-aug):")
    for cls, wi, n in zip(CLASS_NAMES, w, counts):
        print(f"  {cls:20s}: {wi:.3f}  (n={int(n)})")
    return torch.tensor(w, dtype=torch.float32, device=device)


print("Dataset ready.")


Dataset ready.


## Penjelasan Cell Dataset dan DataLoader

Cell ini mendefinisikan class dataset, strategi Patch-based Domain Adaptation untuk kelas tertentu, pembuatan DataLoader, dan skema seed per worker agar hasil lebih reproducible. Output dari bagian ini adalah batch tensor yang siap masuk model.

In [ ]:
"""CELL 5 — Models
Fixes:
  #4  — FSFM uses pretrain_ds_mean_std.txt normalization (not ImageNet)
  #5  — DINOv3 ConvNeXt loaded via official hub with weights= param (strict)
  #6  — FSFM partial load warned loudly; unexpected keys printed
  #11 — all checkpoints report unexpected keys; high missing rate raises error
  #17 — LoRA errors printed, not swallowed silently
  #18 — torch.compile disabled for inference (only used during training)
"""

import sys
import torch
import torch.nn as nn
from pathlib import Path
from typing import Tuple, Set

sys.path.insert(0, str(FSFM_DIR))


# ── Checkpoint validation helper (#11) ─────────────────────────────────────────

def _load_state_strict(model: nn.Module, state: dict, name: str,
                        allow_missing_frac: float = 0.05) -> None:
    """
    Load state dict with strict=False but validate result loudly.
    Raises if >allow_missing_frac of keys are missing or any unexpected keys exist.
    """
    total     = len(model.state_dict())
    missing, unexpected = model.load_state_dict(state, strict=False)

    if unexpected:
        print(f"  [WARN] {name}: {len(unexpected)} unexpected keys: {unexpected[:5]}")
    if missing:
        frac = len(missing) / max(1, total)
        msg  = (f"  {name}: loaded {total-len(missing)}/{total} keys "
                f"({len(missing)} missing, {len(unexpected)} unexpected)")
        if frac > allow_missing_frac:
            raise RuntimeError(
                f"{msg}\n  Missing fraction {frac:.2%} exceeds threshold. "
                "Check that the correct checkpoint file is being used."
            )
        print(msg)
    else:
        print(f"  {name}: all {total} keys loaded ✓")


def _unwrap_state(raw: dict) -> dict:
    """Unwrap common checkpoint wrapper keys."""
    if not isinstance(raw, dict):
        return raw
    for k in ("model", "state_dict", "teacher", "student"):
        if k in raw:
            print(f"  Unwrapping checkpoint key: '{k}'")
            return raw[k]
    return raw


# ── FSFM normalization (#4) ─────────────────────────────────────────────────────

def _load_fsfm_normalization():
    txt_path = FSFM_MEAN_STD_TXT
    if not txt_path.exists():
        print(f"  [WARN] {txt_path} not found. Falling back to ImageNet stats.")
        return (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    try:
        import re
        content = txt_path.read_text().strip()
        # Robust: extract all floats regardless of format
        # Handles: "[0.548, 0.432, 0.386]" or "mean 0.548 0.432 0.386\nstd ..."
        floats = [float(x) for x in re.findall(r'\d+\.\d+(?:e[+-]?\d+)?', content)]
        if len(floats) < 6:
            raise ValueError(f"Expected 6 floats (mean+std), got {len(floats)}: {floats}")
        mean = tuple(floats[:3])
        std  = tuple(floats[3:6])
        print(f"  FSFM normalization: mean={mean}  std={std}")
        return mean, std
    except Exception as e:
        print(f"  [WARN] Failed to parse {txt_path}: {e}. Falling back to ImageNet.")
        return (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

# ── LoRA injection (#17: verbose error handling) ────────────────────────────────

class LoRALinear(nn.Module):
    """Drop-in LoRA wrapper for nn.Linear — no peft dependency."""
    def __init__(self, linear: nn.Linear, r: int, alpha: int, dropout: float):
        super().__init__()
        self.linear   = linear
        self.linear.weight.requires_grad = False
        if linear.bias is not None:
            linear.bias.requires_grad = False
        d_in, d_out   = linear.weight.shape[1], linear.weight.shape[0]
        self.lora_A   = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.lora_B   = nn.Parameter(torch.zeros(d_out, r))
        self.scale    = alpha / r
        self.drop     = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear(x) + self.drop(x) @ self.lora_A.T @ self.lora_B.T * self.scale


def apply_lora(backbone: nn.Module, r: int, alpha: int, dropout: float,
               target_modules: list) -> nn.Module:
    """Inject LoRA into all Linear layers whose name contains a target string."""
    for p in backbone.parameters():
        p.requires_grad = False

    replaced = 0
    for name, module in list(backbone.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        if not any(t in name for t in target_modules):
            continue
        # Navigate to parent and replace
        parts  = name.split(".")
        parent = backbone
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], LoRALinear(module, r, alpha, dropout))
        replaced += 1

    if replaced == 0:
        print("  [WARN] No LoRA targets matched — unfreezing last 4 blocks as fallback")
        blocks = getattr(backbone, "blocks", [])
        for block in list(blocks)[-4:]:
            for p in block.parameters():
                p.requires_grad = True
    else:
        print(f"  LoRA injected into {replaced} linear layers (r={r}, alpha={alpha})")

    n_tr  = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    n_tot = sum(p.numel() for p in backbone.parameters())
    print(f"  Trainable: {n_tr:,}/{n_tot:,} ({100*n_tr/n_tot:.2f}%)")
    return backbone


# ── Shared heads ───────────────────────────────────────────────────────────────

class ClassificationHead(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Dropout(dropout), nn.Linear(in_dim, out_dim)
        )
    def forward(self, x): return self.net(x)


class DualBranchHead(nn.Module):
    def __init__(self, branch_dim, num_classes=NUM_CLASSES):
        super().__init__()
        self.head6    = ClassificationHead(branch_dim * 2, num_classes, 0.3)
        self.head_bin = ClassificationHead(branch_dim * 2, 1, 0.2)
    def forward(self, f_full, f_crop):
        feat = torch.cat([f_full, f_crop], dim=-1)
        return self.head6(feat), self.head_bin(feat)


# ── DINOv3 ViT-H+ ──────────────────────────────────────────────────────────────

class DINOv3ViTHModel(nn.Module):
    """ViT-H+ 32 blocks, hidden=1280. Intermediate CLS at 20+26+final → 3840/branch."""
    INTERMEDIATE = {20, 26}

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        print("Building DINOv3 ViT-H+ ...")

        import os
        os.environ["TORCH_HOME"] = "/dev/shm/torch_hub"
        backbone = torch.hub.load(
            str(DINOV3_DIR), 'dinov3_vith16plus',
            source='local', weights=str(DINOV3_VITH_PT)
        )
        print(f"  Loaded via torch.hub: {DINOV3_VITH_PT.name}")

        self.hidden_size = int(getattr(backbone, "embed_dim", 1280))
        self.backbone    = apply_lora(backbone, cfg.lora_r, cfg.lora_alpha,
                                       cfg.lora_dropout, ["qkv", "q_proj", "v_proj"])
        if cfg.grad_checkpoint:
            for attr in ("use_checkpoint", "set_grad_checkpointing"):
                if hasattr(self.backbone, attr):
                    if callable(getattr(self.backbone, attr)):
                        getattr(self.backbone, attr)(True)
                    else:
                        setattr(self.backbone, attr, True)
                    print(f"  Gradient checkpointing enabled ({attr})")
                    break

        self.head = DualBranchHead(self.hidden_size * 3, NUM_CLASSES)
        self._summary()

    def _summary(self):
        tot = sum(p.numel() for p in self.parameters())
        tr  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  ViT-H+: {tot:,} total | {tr:,} trainable ({100*tr/tot:.2f}%)")

    def _extract(self, x):
        if hasattr(self.backbone, "prepare_tokens_with_masks"):
            tok = self.backbone.prepare_tokens_with_masks(x, masks=None)
        else:
            tok = self.backbone.prepare_tokens(x)
        cls_list = []
        for i, block in enumerate(self.backbone.blocks):
            tok = block(tok)
            if i in self.INTERMEDIATE:
                cls_list.append(tok[:, 0])
        tok = self.backbone.norm(tok)
        cls_list.append(tok[:, 0])
        return torch.cat(cls_list, dim=-1)   # [B, 3840]

    def forward(self, batch):
        ff = self._extract(batch["image_full"].to(DEVICE))
        fc = self._extract(batch["image_crop"].to(DEVICE))
        return self.head(ff, fc)

    def get_param_groups(self, blr, hlr):
        return [
            {"params": [p for p in self.backbone.parameters() if p.requires_grad], "lr": blr},
            {"params": list(self.head.parameters()), "lr": hlr},
        ]


# ── DINOv3 ConvNeXt-Large (#5: official hub loader) ────────────────────────────

class DINOv3ConvNeXtModel(nn.Module):
    """ConvNeXt-L distilled from DINOv3. 6ch (RGB+FFT) via stem weight replication."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        print("Building DINOv3 ConvNeXt-Large ...")

        # Official hub loader handles architecture + strict weight loading (#5)
        backbone = torch.hub.load(
            str(DINOV3_DIR), 'dinov3_convnext_large',
            source='local', weights=str(DINOV3_CNX_PT)
        )
        print(f"  Loaded via torch.hub: {DINOV3_CNX_PT.name}")

        self.hidden_size = int(getattr(backbone, "num_features",
                               getattr(backbone, "embed_dim", 1536)))

        # Expand stem from 3-channel to 6-channel for FFT (#13)
        if cfg.use_fft:
            self._expand_stem_to_6ch(backbone)

        self.backbone = backbone
        self.head     = DualBranchHead(self.hidden_size, NUM_CLASSES)
        self._summary()

    @staticmethod
    def _expand_stem_to_6ch(backbone: nn.Module) -> None:
        """
        Find first Conv2d in stem, expand in_channels 3→6.
        Replicate RGB weights for channels 3-5 (good starting point).
        """
        stem_conv = None
        # Try common paths
        for attr in ("stem", "downsample_layers"):
            mod = getattr(backbone, attr, None)
            if mod is None:
                continue
            first = list(mod.children())[0] if hasattr(mod, "children") else mod
            if isinstance(first, nn.Conv2d):
                stem_conv = first; break
            for m in first.modules():
                if isinstance(m, nn.Conv2d):
                    stem_conv = m; break
            if stem_conv:
                break

        if stem_conv is None:
            print("  [WARN] Could not find stem Conv2d for 6ch expansion")
            return

        old_w = stem_conv.weight.data.clone()   # [out, 3, kH, kW]
        new_w = torch.cat([old_w, old_w], dim=1)  # [out, 6, kH, kW]
        stem_conv.weight = nn.Parameter(new_w)
        stem_conv.in_channels = 6
        print(f"  Stem expanded to 6-channel (RGB weights replicated for FFT channels)")

    def _summary(self):
        tot = sum(p.numel() for p in self.parameters())
        tr  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  ConvNeXt-L: {tot:,} total | {tr:,} trainable ({100*tr/tot:.2f}%)")

    def _extract(self, x):
        return self.backbone(x)   # [B, 1536]

    def forward(self, batch):
        ff = self._extract(batch["image_full"].to(DEVICE))
        fc = self._extract(batch["image_crop"].to(DEVICE))
        return self.head(ff, fc)

    def get_param_groups(self, blr, hlr):
        d = 0.7
        s0, s1, s2, s3, other = [], [], [], [], []
        for name, p in self.backbone.named_parameters():
            if   "stages.0" in name: s0.append(p)
            elif "stages.1" in name: s1.append(p)
            elif "stages.2" in name: s2.append(p)
            elif "stages.3" in name: s3.append(p)
            else:                    other.append(p)
        return [
            {"params": other, "lr": blr * d**4},
            {"params": s0,    "lr": blr * d**3},
            {"params": s1,    "lr": blr * d**2},
            {"params": s2,    "lr": blr * d},
            {"params": s3,    "lr": blr},
            {"params": list(self.head.parameters()), "lr": hlr},
        ]


# ── FSFM ViT-B/16 (#4: correct normalization, #6: loud partial load warning) ───

class FSFMModel(nn.Module):
    """FSFM ViT-B/16. Uses face-dataset normalization from pretrain_ds_mean_std.txt."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        print("Building FSFM ViT-B/16 ...")

        # Checkpoint and norm stats downloaded in cell 2
        ensure_fsfm_assets()
        if not FSFM_CKPT.exists():
            raise FileNotFoundError(
                f"FSFM checkpoint not found at {FSFM_CKPT}. "
                "Re-run cell 2 to download it."
            )

        # Load correct normalization (#4) — updates global FSFM_MEAN/STD in cell 3
        mean, std = _load_fsfm_normalization()
        global FSFM_MEAN, FSFM_STD
        FSFM_MEAN = mean
        FSFM_STD  = std
        print(f"  Normalization set: mean={mean}  std={std}")

        # Build architecture from FSFM repo (#6: explicit path check)
        arch_file = FSFM_DIR / "models_vit.py"
        if not arch_file.exists():
            raise FileNotFoundError(
                f"FSFM models_vit.py not found at {arch_file}. "
                "Was the repo cloned correctly in cell 1?"
            )
        try:
            import models_vit
            backbone = models_vit.__dict__["vit_base_patch16"](
                num_classes=0, drop_path_rate=0.1, global_pool=True,
            )
            print("  Architecture: FSFM models_vit.vit_base_patch16")
        except Exception as e:
            raise RuntimeError(f"Failed to import FSFM models_vit: {e}")

        # Load weights with strict validation (#6, #11)
        state = torch.load(str(FSFM_CKPT), map_location="cpu")
        state = _unwrap_state(state)
        _load_state_strict(backbone, state, "FSFM", allow_missing_frac=0.10)

        self.hidden_size = 768
        self.backbone    = apply_lora(backbone, cfg.lora_r, cfg.lora_alpha,
                                       cfg.lora_dropout, ["qkv", "q", "k", "v"])
        if cfg.grad_checkpoint:
            if hasattr(self.backbone, "use_checkpoint"):
                self.backbone.use_checkpoint = True
                print("  Gradient checkpointing enabled")

        self.head = DualBranchHead(self.hidden_size, NUM_CLASSES)
        self._summary()

    def _summary(self):
        tot = sum(p.numel() for p in self.parameters())
        tr  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  FSFM ViT-B: {tot:,} total | {tr:,} trainable ({100*tr/tot:.2f}%)")

    def _extract(self, x):
        out = self.backbone(x)
        if isinstance(out, torch.Tensor): return out
        if hasattr(out, "last_hidden_state"): return out.last_hidden_state[:, 0]
        return out[0][:, 0] if isinstance(out, (tuple, list)) else out

    def forward(self, batch):
        ff = self._extract(batch["image_full"].to(DEVICE))
        fc = self._extract(batch["image_crop"].to(DEVICE))
        return self.head(ff, fc)

    def get_param_groups(self, blr, hlr):
        return [
            {"params": [p for p in self.backbone.parameters() if p.requires_grad], "lr": blr},
            {"params": list(self.head.parameters()), "lr": hlr},
        ]


# ── Factory ────────────────────────────────────────────────────────────────────

def build_model(cfg) -> nn.Module:
    if cfg.name == "dinov3_vith": return DINOv3ViTHModel(cfg)
    if cfg.name == "dinov3_cnx":  return DINOv3ConvNeXtModel(cfg)
    if cfg.name == "fsfm":        return FSFMModel(cfg)
    raise ValueError(f"Unknown model: {cfg.name}")


def save_checkpoint(model: nn.Module, epoch: int, val_f1: float, path: Path):
    state = model._orig_mod.state_dict() if hasattr(model, "_orig_mod") else model.state_dict()
    torch.save({"epoch": epoch, "val_f1": val_f1, "model_state": state}, path)
    print(f"  Saved: {path.name}  val_f1={val_f1:.4f}")


def load_for_inference(ckpt_path: Path, cfg) -> nn.Module:
    """Build → load → NO compile for inference. (#18: compile overhead not worth it)"""
    model = build_model(cfg).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    _load_state_strict(model, ckpt["model_state"], ckpt_path.name)
    model.eval()
    print(f"  epoch={ckpt.get('epoch','?')} f1={ckpt.get('val_f1','?')}")
    return model


print("Models ready: DINOv3ViTH, DINOv3ConvNeXt, FSFM")


Models ready: DINOv3ViTH, DINOv3ConvNeXt, FSFM


## Penjelasan Cell Definisi Model

Cell ini berisi seluruh definisi arsitektur model, mekanisme LoRA, validasi loading checkpoint, penyesuaian normalisasi FSFM, serta factory function untuk build model training dan inferensi. Bagian ini memastikan bobot pretrain terpasang dengan aman sebelum proses belajar dimulai.

In [ ]:
"""CELL 6 — Training
Updated to use split transform pipeline (spatial_tf + norm_fn) from cell 3.
torch.compile used during training only. (#18)
"""

import math
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict, Optional
import gc

REAL_IDX = CLASS_TO_IDX["realperson"]

def to_binary(labels):
    return (labels == REAL_IDX).float().unsqueeze(-1)

def macro_f1_np(probs, labels):
    return f1_score(labels, probs.argmax(1), average="macro", zero_division=0)

def per_class_f1_np(probs, labels):
    sc = f1_score(labels, probs.argmax(1), average=None, zero_division=0)
    return {CLASS_NAMES[i]: round(float(sc[i]), 4) for i in range(NUM_CLASSES)}

def make_cosine_scheduler(opt, total_steps, warmup_ratio):
    from torch.optim.lr_scheduler import LambdaLR
    warmup = int(total_steps * warmup_ratio)
    def fn(step):
        if step < warmup: return step / max(1, warmup)
        prog = (step - warmup) / max(1, total_steps - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * prog)))
    return LambdaLR(opt, fn)


def _get_norm_fn(cfg):
    """Return the correct normalization function for this model config."""
    if cfg.name == "fsfm":
        return apply_fsfm_norm     # from cell 3, updated after FSFM init
    return apply_imagenet_norm     # from cell 3


def train_one_epoch(model, loader, opt, sched, crit6, crit_bin, bin_w):
    model.train()
    total, steps = 0.0, 0
    opt.zero_grad()
    for step_idx, batch in enumerate(tqdm(loader, desc="  train", leave=False), start=1):
        labs = batch["label"].to(DEVICE)
        bins = to_binary(labs)
        with autocast_ctx():
            l6, lbin = model(batch)
            loss = (crit6(l6, labs) + bin_w * crit_bin(lbin, bins))
        loss.backward()
        steps += 1
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad(); sched.step()
        total += loss.item()
        if SMOKE_TRAIN_STEPS and step_idx >= SMOKE_TRAIN_STEPS:
            print(f"  [SMOKE] stopping training epoch after {step_idx} step(s)")
            break
    return total / max(1, steps)


@torch.no_grad()
def validate(model, loader, crit6, crit_bin, bin_w):
    model.eval()
    all_probs, all_labels = [], []
    total = 0.0
    for batch in tqdm(loader, desc="  val  ", leave=False):
        labs = batch["label"].to(DEVICE)
        bins = to_binary(labs)
        with autocast_ctx():
            l6, lbin = model(batch)
            loss = crit6(l6, labs) + bin_w * crit_bin(lbin, bins)
        all_probs.append(torch.softmax(l6.float(), -1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())
        total += loss.item()
    return np.vstack(all_probs), np.array(all_labels), total / len(loader)

def train_model(cfg, run_name: Optional[str] = None) -> None:
    run_name = run_name or cfg.name
    set_seed()
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    ckpt_path = CKPT_DIR / f"{run_name}.pt"

    # Build model FIRST so FSFM can set the correct normalization stats
    print(f"\n{'='*60}\nTraining: {run_name}\n{'='*60}")
    model    = build_model(cfg).to(DEVICE)
    norm_fn  = _get_norm_fn(cfg)    # correct norm after model init updates FSFM_MEAN/STD

    # Split
    original = build_train_samples(TRAIN_DIR)
    labels   = np.array([l for _, l in original])
    sss      = StratifiedShuffleSplit(1, test_size=VAL_FRACTION, random_state=SEED)
    tr_idx, va_idx = next(sss.split(np.zeros(len(labels)), labels))
    train_orig = [original[i] for i in tr_idx]
    val_orig   = [original[i] for i in va_idx]
    print(f"Split → train: {len(train_orig)}  val: {len(val_orig)}")

    # Preprocess
    cache      = CACHE_DIR / str(cfg.resolution)
    path_map   = preprocess_all(train_orig + val_orig, cache, cfg.resolution,
                                max_workers=NUM_WORKERS)
    train_cached = [(path_map[str(p)], l) for p, l in train_orig]
    val_cached   = [(path_map[str(p)], l) for p, l in val_orig]

    # Augment — source = original images (before resize)
    aug_sub     = AUG_DIR / str(cfg.resolution)
    if SMOKE_TRAIN_STEPS:
        print("[SMOKE] skipping offline augmentation")
        aug_samples = []
    else:
        aug_samples = run_offline_augmentation(train_orig, aug_sub, cfg.resolution,
                                               max_workers=NUM_WORKERS)
    train_samples = train_cached + aug_samples
    print(f"Train: {len(train_samples)} ({len(train_cached)} orig + {len(aug_samples)} aug)")

    # PDA
    real_full = [fp for (fp, _), l in train_samples if l == REAL_IDX]

    # Original path stems for submission ID tracking
    tr_orig_paths = [str(p) for p, _ in train_orig] + \
                    [str(fp) for (fp, _), _ in aug_samples]
    va_orig_paths = [str(p) for p, _ in val_orig]

    # Loaders
    tr_tf  = get_spatial_transform(cfg.resolution, augment=True)
    va_tf  = get_spatial_transform(cfg.resolution, augment=False)

    tr_loader = make_loader(train_samples, tr_tf, norm_fn,
                            shuffle=True, batch_size=cfg.batch_size,
                            is_train=True, use_fft=cfg.use_fft,
                            pda_real_full=real_full,
                            orig_paths=tr_orig_paths, drop_last=True)
    va_loader = make_loader(val_cached, va_tf, norm_fn,
                            shuffle=False, batch_size=cfg.batch_size * 2,
                            use_fft=cfg.use_fft,
                            orig_paths=va_orig_paths)

    # Optimizer + scheduler
    param_groups = model.get_param_groups(cfg.backbone_lr, cfg.head_lr)
    opt          = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay, eps=1e-8)
    total_steps  = len(tr_loader) * cfg.epochs
    sched        = make_cosine_scheduler(opt, total_steps, cfg.warmup_ratio)

    # Compile for training (#18: compile only during training, not inference)
    if ENABLE_COMPILE:
        print("Compiling model for training ...")
        model = torch.compile(model, mode="default")
    else:
        print("Skipping torch.compile for this runtime")

    # Losses
    cw        = mild_class_weights(train_samples, DEVICE)
    crit6     = nn.CrossEntropyLoss(weight=cw, label_smoothing=cfg.label_smoothing)
    n_real    = sum(1 for _, l in train_samples if l == REAL_IDX)
    n_spoof   = len(train_samples) - n_real
    pos_w     = torch.tensor([n_spoof / max(1, n_real)], device=DEVICE).clamp(1.0, 8.0)
    crit_bin  = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    print(f"Binary pos_weight: {pos_w.item():.3f}")
    if DEVICE.type == "cuda":
        torch.backends.cudnn.benchmark = True

    best_f1, best_ep, overfit_c, noimprove_c = 0.0, 0, 0, 0

    for epoch in range(cfg.epochs):
        tr_loss = train_one_epoch(model, tr_loader, opt, sched,
                                  crit6, crit_bin, cfg.binary_weight)
        probs, labels_v, va_loss = validate(model, va_loader,
                                            crit6, crit_bin, cfg.binary_weight)
        val_f1 = macro_f1_np(probs, labels_v)

        print(f"Ep {epoch+1:03d}/{cfg.epochs}  "
              f"tr={tr_loss:.4f}  va={va_loss:.4f}  f1={val_f1:.4f}"
              + (" ✓" if val_f1 > best_f1 else ""))

        if (epoch + 1) % 5 == 0:
            for cls, s in per_class_f1_np(probs, labels_v).items():
                print(f"    {cls:20s}: {s:.4f}")

        if val_f1 > best_f1:
            best_f1, best_ep, overfit_c, noimprove_c = val_f1, epoch+1, 0, 0
            save_checkpoint(model, epoch+1, val_f1, ckpt_path)
        else:
            noimprove_c += 1
            overfit_c = overfit_c + 1 if val_f1 < (best_f1 - cfg.overfit_delta) else 0
            if overfit_c >= cfg.early_stop_patience:
                print(f"Early stop (overfit). Best={best_f1:.4f} @ ep {best_ep}"); break
            if noimprove_c >= cfg.early_stop_patience * 2:
                print(f"Early stop (plateau). Best={best_f1:.4f} @ ep {best_ep}"); break

    print(f"\n{run_name} DONE  best_f1={best_f1:.4f} @ epoch {best_ep}")
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Training ready.")


Training ready.


## Penjelasan Cell Training

Cell ini menjalankan loop training lengkap: split train-validasi, preprocessing cache, augmentasi offline, pembuatan loader, optimizer, scheduler cosine, evaluasi macro-F1, early stopping, dan penyimpanan checkpoint terbaik. Gunakan cell ini untuk melatih satu model sesuai konfigurasi yang dipilih.

In [ ]:
"""CELL 7 — Inference + Ensemble + Submission
Fixes:
  #1  — write_submission unpacks (full_p, crop_p) correctly
  #2  — submission IDs use ORIGINAL test file stems, not cached file names
  #12 — FFT computed once per image before TTA loop (not repeated per view)
  #18 — no torch.compile at inference
  #19 — ensemble export is submission-only and does not score hidden labels
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from tqdm.auto import tqdm


def build_tta_spatial_transforms(resolution: int) -> List[A.Compose]:
    """Three spatial-only TTA views (no normalize). (#13 consistency)"""
    big  = int(resolution * 1.15)
    base = [ToTensorV2()]
    return [
        A.Compose([A.Resize(resolution, resolution)] + base),
        A.Compose([A.Resize(resolution, resolution), A.HorizontalFlip(p=1.0)] + base),
        A.Compose([A.Resize(big, big), A.CenterCrop(resolution, resolution)] + base),
    ]

@torch.no_grad()
def infer_model(
    model,
    test_samples,
    orig_stems,
    cfg,
    batch_size: int = 32,
    n_tta: int = 3,
):
    model.eval()
    tta_tfs = build_tta_spatial_transforms(cfg.resolution)[:n_tta]
    norm_fn = apply_fsfm_norm if cfg.name == "fsfm" else apply_imagenet_norm
    n = len(test_samples)
    print(f"  Loading {n} test images ...")
    full_imgs = [load_image(Path(fp)) for (fp, _), _ in tqdm(test_samples, leave=False)]
    crop_imgs = [load_image(Path(cp)) for (_, cp), _ in tqdm(test_samples, leave=False)]
    if cfg.use_fft:
        print("  Precomputing FFT channels ...")
        import cv2 as _cv2
        def _to_01(img): return torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        def _fft_for(img):
            r = _cv2.resize(img, (cfg.resolution, cfg.resolution), interpolation=_cv2.INTER_LANCZOS4)
            return compute_fft_channels(_to_01(r))
        full_fft = [_fft_for(img) for img in tqdm(full_imgs, desc="  FFT full", leave=False)]
        crop_fft = [_fft_for(img) for img in tqdm(crop_imgs, desc="  FFT crop", leave=False)]
    all_tta_probs, all_tta_bins = [], []
    for ti, tf in enumerate(tta_tfs):
        view_probs, view_bins = [], []
        for start in range(0, n, batch_size):
            end = start + batch_size
            b_full = [tf(image=img)["image"].float() / 255.0 for img in full_imgs[start:end]]
            b_crop = [tf(image=img)["image"].float() / 255.0 for img in crop_imgs[start:end]]
            b_full_n = [norm_fn(t) for t in b_full]
            b_crop_n = [norm_fn(t) for t in b_crop]
            if cfg.use_fft:
                bf = torch.stack([torch.cat([n_, f_], dim=0) for n_, f_ in zip(b_full_n, full_fft[start:end])]).to(DEVICE)
                bc = torch.stack([torch.cat([n_, f_], dim=0) for n_, f_ in zip(b_crop_n, crop_fft[start:end])]).to(DEVICE)
            else:
                bf = torch.stack(b_full_n).to(DEVICE)
                bc = torch.stack(b_crop_n).to(DEVICE)
            with autocast_ctx():
                l6, lbin = model({"image_full": bf, "image_crop": bc})
            view_probs.append(torch.softmax(l6.float(), -1).cpu().numpy())
            view_bins.append(torch.sigmoid(lbin.float()).cpu().numpy())
        all_tta_probs.append(np.vstack(view_probs))
        all_tta_bins.append(np.vstack(view_bins))
        print(f"  TTA {ti+1}/{n_tta} done")
    probs = np.mean(all_tta_probs, axis=0)
    bin_pred = np.mean(all_tta_bins, axis=0)
    pred6 = probs.argmax(1)
    r6 = pred6 == REAL_IDX
    rbin = bin_pred[:, 0] >= 0.5
    disagree = r6 != rbin
    if disagree.any():
        print(f"\n[Diagnostic] {disagree.sum()} binary/6-class disagreements:")
        for i in np.where(disagree)[0][:10]:
            p6 = IDX_TO_CLASS[int(pred6[i])]
            pb = "realperson" if rbin[i] else "spoof"
            conf = probs[i, pred6[i]]
            print(f"  {orig_stems[i]:30s} 6cls={p6:20s}({conf:.2f}) bin={pb}")
    else:
        print("[Diagnostic] Binary and 6-class fully agree. ✓")
    return probs, orig_stems

def write_submission(
    orig_stems: List[str],
    probs:      np.ndarray,
    path:       Path,
) -> None:
    """Write submission CSV. IDs = original test image stems. (#1, #2)"""
    preds = probs.argmax(1)
    rows  = [{"id": stem, "label": IDX_TO_CLASS[int(p)]}
             for stem, p in zip(orig_stems, preds)]
    df    = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Submission: {path}  ({len(df)} rows)")
    for cls in CLASS_NAMES:
        print(f"  {cls:20s}: {df['label'].eq(cls).sum()}")


def ensemble_submit(
    checkpoints: Dict[str, Tuple[Path, object]],
    run_name:    str = "ensemble",
    n_tta:       int = 3,
) -> Optional[Path]:
    test_orig = build_test_samples(TEST_DIR)
    if not test_orig:
        print(f"[SKIP] No test images found under {TEST_DIR}")
        return None
    orig_stems = [p.stem for p, _ in test_orig]

    all_probs = []

    for name, (ckpt_path, cfg) in checkpoints.items():
        if not ckpt_path.exists():
            print(f"[SKIP] {name}: {ckpt_path} not found")
            continue
        print()
        print(f"── {name} ──")
        cache    = CACHE_DIR / str(cfg.resolution)
        path_map = preprocess_all(test_orig, cache, cfg.resolution,
                                  max_workers=NUM_WORKERS)
        test_cached = [(path_map[str(p)], l) for p, l in test_orig]

        model = load_for_inference(ckpt_path, cfg)
        probs, _ = infer_model(model, test_cached, orig_stems, cfg, n_tta=n_tta)
        all_probs.append(probs)
        np.save(CKPT_DIR / f"probs_{name}.npy", probs)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not all_probs:
        print("ERROR: no checkpoints")
        return None

    ensemble = np.mean(all_probs, axis=0)
    sub_path = CKPT_DIR / f"submission_{run_name}.csv"
    write_submission(orig_stems, ensemble, sub_path)
    return sub_path


print("Inference ready.")


Inference ready.


## Penjelasan Cell Inferensi dan Ensemble

Cell ini digunakan untuk inferensi data test dengan TTA, penggabungan probabilitas antar-model, dan ekspor file submission. Bagian ini juga menyiapkan diagnosis ketidaksesuaian prediksi biner versus 6 kelas untuk membantu analisis hasil.

In [ ]:
"""CELL 8 — Run
Unzip dataset, train all three models sequentially, ensemble submit.
Each model trains independently — if one crashes, the others still run.
"""

import zipfile, gc

print(f"TRAIN_DIR: {TRAIN_DIR}  exists={TRAIN_DIR.exists()}")

print(f"TEST_DIR:  {TEST_DIR}  exists={TEST_DIR.exists()}")

# ── Train all three models ─────────────────────────────────────────────────────

results = {}
run_filter = {
    name.strip()
    for name in os.environ.get("FSFM_RUN_MODELS", "dinov3_cnx,fsfm,dinov3_vith").split(",")
    if name.strip()
}
train_jobs = [
    ("dinov3_cnx", "DINOv3 ConvNeXt-Large (RGB+FFT, dual branch)", DINOV3_CNX_CFG),
    ("fsfm", "FSFM ViT-B/16 (face-specific pretraining)", FSFM_CFG),
    ("dinov3_vith", "DINOv3 ViT-H+", DINOV3_VITH_CFG),
]

for idx, (name, title, cfg) in enumerate(train_jobs, start=1):
    if name not in run_filter:
        print(f"[SKIP] {name} excluded by FSFM_RUN_MODELS")
        continue
    print("
" + "="*60)
    print(f"TRAINING {idx}/{len(train_jobs)} — {title}")
    print("="*60)
    try:
        train_model(cfg, run_name=name)
        results[name] = CKPT_DIR / f"{name}.pt"
    except Exception as e:
        print(f"[ERROR] {name}: {e}")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("
" + "="*60)
print("TRAINING COMPLETE")
for name, path in results.items():
    print(f"  {name:20s}: {path}  exists={path.exists()}")
print("="*60)

# ── Ensemble submit ────────────────────────────────────────────────────────────

checkpoints = {
    name: (path, {"dinov3_vith": DINOV3_VITH_CFG,
                  "dinov3_cnx":  DINOV3_CNX_CFG,
                  "fsfm":        FSFM_CFG}[name])
    for name, path in results.items() if path.exists()
}

if checkpoints:
    print(f"
Ensembling {len(checkpoints)} model(s): {list(checkpoints.keys())}")
    submission_path = ensemble_submit(checkpoints, run_name="final_ensemble", n_tta=3)
    print(f"
Final ensemble submission: {submission_path}")
else:
    print("No checkpoints found — nothing to ensemble.")


TRAIN_DIR: /content/dataset_rotated  exists=True
TEST_DIR:  /content/drive/MyDrive/data_dac/datatest/test_images  exists=True

TRAINING 1/3 — DINOv3 ConvNeXt-Large (RGB+FFT, dual branch)

Training: dinov3_cnx
Building DINOv3 ConvNeXt-Large ...
Downloading: "file:///content/drive/MyDrive/faris/dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth" to /dev/shm/torch_hub/hub/checkpoints/dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth


100%|██████████| 781M/781M [00:00<00:00, 1.25GB/s]


  Loaded via torch.hub: dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth
  Stem expanded to 6-channel (RGB weights replicated for FFT channels)
  ConvNeXt-L: 196,273,351 total | 196,273,351 trainable (100.00%)
Split → train: 1056  val: 265
Cache: all 1321 present at /dev/shm/fas_cache/384
Augmenting 997 → /dev/shm/fas_aug/384 (4 threads, RAM)...
  Done: 997/997
Train: 2053 (1056 orig + 997 aug)
Skipping torch.compile for this runtime
Class weights (post-aug):
  fake_mannequin      : 0.762  (n=435)
  fake_mask           : 0.916  (n=362)
  fake_printed        : 0.884  (n=375)
  fake_screen         : 0.978  (n=339)
  fake_unknown        : 1.321  (n=251)
  realperson          : 1.139  (n=291)
Binary pos_weight: 6.055


  train:   0%|          | 0/128 [00:00<?, ?it/s]

  val  :   0%|          | 0/9 [00:00<?, ?it/s]

...

Ep 033/60  tr=0.4328  va=0.6393  f1=0.9551


  train:   0%|          | 0/128 [00:00<?, ?it/s]

  val  :   0%|          | 0/9 [00:00<?, ?it/s]

Ep 034/60  tr=0.4326  va=0.5782  f1=0.9610


  train:   0%|          | 0/128 [00:00<?, ?it/s]

## Penjelasan Cell Eksekusi End-to-End

Cell ini menjalankan pipeline utama secara berurutan: training model yang dipilih melalui variabel lingkungan, pembersihan memori antar-model, lalu ensemble otomatis untuk membuat submission akhir. Jalankan cell ini jika ingin proses penuh sekali jalan.

In [ ]:
# FSFM forward hotfix + continue FSFM and DINOv3 ViT-H + reuse existing ConvNeXt for ensemble
import sys
import gc
import traceback
import torch
# trusted local checkpoint load fix for torch >= 2.6
if not hasattr(torch, "_orig_load_continue_hotfix_v2"):
    torch._orig_load_continue_hotfix_v2 = torch.load
    def _torch_load_continue_hotfix_v2(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return torch._orig_load_continue_hotfix_v2(*args, **kwargs)
    torch.load = _torch_load_continue_hotfix_v2
# FSFM hotfix: bypass timm forward_head/global_pool path
def _fsfm_extract_hotfix_v2(self, x):
    if hasattr(self.backbone, "forward_features"):
        out = self.backbone.forward_features(x)
    else:
        out = self.backbone(x)
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state[:, 0]
    return out[0][:, 0] if isinstance(out, (tuple, list)) else out
FSFMModel._extract = _fsfm_extract_hotfix_v2
print("Applied FSFM _extract hotfix: use backbone.forward_features() directly")
results = {}
# keep existing ConvNeXt checkpoint
cnx_ckpt = CKPT_DIR / "dinov3_cnx.pt"
if cnx_ckpt.exists():
    results["dinov3_cnx"] = cnx_ckpt
    print(f"Using existing dinov3_cnx checkpoint: {cnx_ckpt}")
else:
    print(f"[WARN] dinov3_cnx checkpoint not found: {cnx_ckpt}")
# continue remaining models
for name, title, cfg in [
    ("fsfm", "FSFM ViT-B/16 (face-specific pretraining)", FSFM_CFG),
    ("dinov3_vith", "DINOv3 ViT-H+", DINOV3_VITH_CFG),
]:
    print("\n" + "=" * 60)
    print(f"TRAINING — {title}")
    print("=" * 60)
    try:
        train_model(cfg, run_name=name)
        ckpt = CKPT_DIR / f"{name}.pt"
        if ckpt.exists():
            results[name] = ckpt
    except Exception as e:
        msg = str(e).strip()
        print(f"[ERROR] {name}: {type(e).__name__}: {msg if msg else repr(e)}")
        traceback.print_exc(limit=10)
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print("\n" + "=" * 60)
print("AVAILABLE CHECKPOINTS")
for name, path in results.items():
    print(f"  {name:20s}: {path}  exists={path.exists()}")
print("=" * 60)
checkpoints = {
    name: (path, {
        "dinov3_cnx": DINOV3_CNX_CFG,
        "fsfm": FSFM_CFG,
        "dinov3_vith": DINOV3_VITH_CFG,
    }[name])
    for name, path in results.items()
    if path.exists()
}
if checkpoints:
    print(f"\nEnsembling {len(checkpoints)} model(s): {list(checkpoints.keys())}")
    f1 = ensemble_submit(checkpoints, run_name="final_ensemble_resume_v2", n_tta=3)
    print(f"\nFinal ensemble actual_test Macro F1: {f1:.4f}")
else:
    print("No checkpoints found — nothing to ensemble.")

Applied FSFM _extract hotfix: use backbone.forward_features() directly
Using existing dinov3_cnx checkpoint: /content/drive/MyDrive/faris/checkpoint/dinov3_cnx.pt

TRAINING — FSFM ViT-B/16 (face-specific pretraining)

Training: fsfm
Building FSFM ViT-B/16 ...
FSFM checkpoint: fsfm_vit_b_vf2_400e.pth ✓
FSFM norm stats:  fsfm_pretrain_mean_std.txt ✓
  FSFM normalization: mean=(0.5482207536697388, 0.42340534925460815, 0.3654651641845703)  std=(0.2789176106452942, 0.2438540756702423, 0.23493893444538116)
  Normalization set: mean=(0.5482207536697388, 0.42340534925460815, 0.3654651641845703)  std=(0.2789176106452942, 0.2438540756702423, 0.23493893444538116)
  Architecture: FSFM models_vit.vit_base_patch16
  Unwrapping checkpoint key: 'model'
  [WARN] FSFM: 149 unexpected keys: ['mask_token', 'rep_decoder_pos_embed', 'decoder_pos_embed', 'norm.weight', 'norm.bias']
  FSFM: loaded 148/150 keys (2 missing, 149 unexpected)
  LoRA injected into 48 linear layers (r=8, alpha=16)
  Trainable: 1,179

  train:   0%|          | 0/64 [00:00<?, ?it/s]

  val  :   0%|          | 0/5 [00:00<?, ?it/s]

Ep 001/60  tr=2.3203  va=2.2799  f1=0.2202 ✓
  Saved: fsfm.pt  val_f1=0.2202


  train:   0%|          | 0/64 [00:00<?, ?it/s]

...

  TTA 1/3 done
  TTA 2/3 done
  TTA 3/3 done

[Diagnostic] 5 binary/6-class disagreements:
  test_122                       6cls=realperson          (0.61) bin=spoof
  test_200                       6cls=realperson          (0.49) bin=spoof
  test_207                       6cls=realperson          (0.58) bin=spoof
  test_343                       6cls=realperson          (0.56) bin=spoof
  test_348                       6cls=realperson          (0.59) bin=spoof

── fsfm ──
Preprocessing 404 → /dev/shm/fas_cache/224 (4 threads)
  Done: 404/404
Building FSFM ViT-B/16 ...
FSFM checkpoint: fsfm_vit_b_vf2_400e.pth ✓
FSFM norm stats:  fsfm_pretrain_mean_std.txt ✓
  FSFM normalization: mean=(0.5482207536697388, 0.42340534925460815, 0.3654651641845703)  std=(0.2789176106452942, 0.2438540756702423, 0.23493893444538116)
  Normalization set: mean=(0.5482207536697388, 0.42340534925460815, 0.3654651641845703)  std=(0.2789176106452942, 0.2438540756702423, 0.23493893444538116)
  Architecture: FSFM mo

  0%|          | 0/404 [00:00<?, ?it/s]

  0%|          | 0/404 [00:00<?, ?it/s]

  TTA 1/3 done
  TTA 2/3 done
  TTA 3/3 done

[Diagnostic] 60 binary/6-class disagreements:
  test_003                       6cls=fake_mask           (0.66) bin=realperson
  test_007                       6cls=fake_mask           (0.46) bin=realperson
  test_009                       6cls=fake_mask           (0.47) bin=realperson
  test_016                       6cls=fake_mannequin      (0.37) bin=realperson
  test_028                       6cls=fake_printed        (0.57) bin=realperson
  test_034                       6cls=fake_mask           (0.81) bin=realperson
  test_036                       6cls=fake_screen         (0.67) bin=realperson
  test_044                       6cls=fake_screen         (0.76) bin=realperson
  test_054                       6cls=fake_printed        (0.36) bin=realperson
  test_056                       6cls=fake_screen         (0.68) bin=realperson
Submission: /content/drive/MyDrive/faris/checkpoint/submission_final_ensemble_resume_v2.csv  (404 rows)
  fa

ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

## Penjelasan Cell Resume Training dengan Hotfix FSFM

Cell ini dipakai saat training terhenti dan Anda ingin melanjutkan dari checkpoint yang sudah ada. Isinya termasuk hotfix untuk kompatibilitas loading checkpoint serta patch forward extractor FSFM sebelum melatih model yang tersisa.

In [ ]:
# dinov3_vith only: compatibility hotfixes + micro-batch 4 + grad accumulation 4
import gc
import math
import traceback
import torch
import torch.nn as nn
from tqdm.auto import tqdm
DINOV3_VITH_CFG.batch_size = 4
GRAD_ACCUM_STEPS = 4   # effective batch = 4 * 4 = 16
# LoRA wrapper compatibility with current DINOv3 attention code
LoRALinear.in_features  = property(lambda self: self.linear.in_features)
LoRALinear.out_features = property(lambda self: self.linear.out_features)
LoRALinear.weight       = property(lambda self: self.linear.weight)
LoRALinear.bias         = property(lambda self: self.linear.bias)
LoRALinear.bias_mask    = property(lambda self: getattr(self.linear, "bias_mask", None))
# Current DINOv3 ViT-H token/RoPE path
def _dinov3_vith_extract_hotfix_v3(self, x):
    if hasattr(self.backbone, "prepare_tokens_with_masks"):
        tok, hw = self.backbone.prepare_tokens_with_masks(x, masks=None)
    else:
        tok = self.backbone.prepare_tokens(x)
        hw = None
    rope = None
    if hw is not None and getattr(self.backbone, "rope_embed", None) is not None:
        H, W = hw
        rope = self.backbone.rope_embed(H=H, W=W)
    cls_list = []
    for i, block in enumerate(self.backbone.blocks):
        tok = block(tok, rope)
        if i in self.INTERMEDIATE:
            cls_list.append(tok[:, 0])
    tok = self.backbone.norm(tok)
    cls_list.append(tok[:, 0])
    return torch.cat(cls_list, dim=-1)
DINOv3ViTHModel._extract = _dinov3_vith_extract_hotfix_v3
# Scheduler adjusted for accumulation so effective optimization schedule stays aligned
def make_cosine_scheduler(opt, total_steps, warmup_ratio):
    from torch.optim.lr_scheduler import LambdaLR
    total_steps = math.ceil(total_steps / GRAD_ACCUM_STEPS)
    warmup = int(total_steps * warmup_ratio)
    def fn(step):
        if step < warmup:
            return step / max(1, warmup)
        prog = (step - warmup) / max(1, total_steps - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * prog)))
    return LambdaLR(opt, fn)
# Gradient accumulation train loop
def train_one_epoch(model, loader, opt, sched, crit6, crit_bin, bin_w):
    model.train()
    total, steps = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for step_idx, batch in enumerate(tqdm(loader, desc="  train", leave=False), start=1):
        labs = batch["label"].to(DEVICE)
        bins = to_binary(labs)
        with autocast_ctx():
            l6, lbin = model(batch)
            loss = crit6(l6, labs) + bin_w * crit_bin(lbin, bins)
        (loss / GRAD_ACCUM_STEPS).backward()
        total += loss.item()
        steps += 1
        should_step = (step_idx % GRAD_ACCUM_STEPS == 0) or (step_idx == len(loader))
        if should_step:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            opt.zero_grad(set_to_none=True)
            sched.step()
        if SMOKE_TRAIN_STEPS and step_idx >= SMOKE_TRAIN_STEPS:
            print(f"  [SMOKE] stopping training epoch after {step_idx} step(s)")
            break
    return total / max(1, steps)
print("Applied dinov3_vith hotfixes")
print(f"dinov3_vith batch_size={DINOV3_VITH_CFG.batch_size}, grad_accum={GRAD_ACCUM_STEPS}, effective_batch={DINOV3_VITH_CFG.batch_size * GRAD_ACCUM_STEPS}")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
try:
    train_model(DINOV3_VITH_CFG, run_name="dinov3_vith")
except Exception as e:
    msg = str(e).strip()
    print(f"[ERROR] dinov3_vith: {type(e).__name__}: {msg if msg else repr(e)}")
    traceback.print_exc(limit=10)
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Applied dinov3_vith hotfixes
dinov3_vith batch_size=4, grad_accum=4, effective_batch=16

Training: dinov3_vith
Building DINOv3 ViT-H+ ...
Downloading: "file:///content/drive/MyDrive/faris/dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth" to /dev/shm/torch_hub/hub/checkpoints/dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth


100%|██████████| 3.13G/3.13G [01:04<00:00, 52.0MB/s]


  Loaded via torch.hub: dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth
  LoRA injected into 32 linear layers (r=8, alpha=16)
  Trainable: 1,310,720/841,944,320 (0.16%)
  ViT-H+: 842,028,807 total | 1,395,207 trainable (0.17%)
Split → train: 1056  val: 265
Preprocessing 1321 → /dev/shm/fas_cache/448 (4 threads)
  Done: 1321/1321
Augmenting 1346 → /dev/shm/fas_aug/448 (4 threads, RAM)...


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


  Done: 1346/1346
Train: 2402 (1056 orig + 1346 aug)
Skipping torch.compile for this runtime
Class weights (post-aug):
  fake_mannequin      : 0.641  (n=435)
  fake_mask           : 2.177  (n=128)
  fake_printed        : 0.290  (n=960)
  fake_screen         : 0.829  (n=336)
  fake_unknown        : 1.106  (n=252)
  realperson          : 0.957  (n=291)
Binary pos_weight: 7.254


...

  val  :   0%|          | 0/34 [00:00<?, ?it/s]

Ep 047/60  tr=0.6256  va=0.5481  f1=0.9800


  train:   0%|          | 0/600 [00:00<?, ?it/s]

  val  :   0%|          | 0/34 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Penjelasan Cell Hotfix Khusus DINOv3 ViT-H

Cell ini fokus untuk skenario memory-constrained pada DINOv3 ViT-H: micro-batch, gradient accumulation, patch kompatibilitas LoRA, dan penyesuaian scheduler. Gunakan hanya bila training ViT-H perlu stabilisasi tambahan.

In [ ]:

# Final submission only: load 3 checkpoints, ensemble, write CSV, skip evaluation
import gc
import math
import traceback
import numpy as np
import torch
from pathlib import Path
# Trusted local checkpoint loading for torch >= 2.6
if not hasattr(torch, "_orig_load_submit_only"):
    torch._orig_load_submit_only = torch.load
    def _torch_load_submit_only(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return torch._orig_load_submit_only(*args, **kwargs)
    torch.load = _torch_load_submit_only
# LoRA wrapper compatibility for current DINOv3 attention code
LoRALinear.in_features  = property(lambda self: self.linear.in_features)
LoRALinear.out_features = property(lambda self: self.linear.out_features)
LoRALinear.weight       = property(lambda self: self.linear.weight)
LoRALinear.bias         = property(lambda self: self.linear.bias)
LoRALinear.bias_mask    = property(lambda self: getattr(self.linear, "bias_mask", None))
# FSFM inference hotfix: bypass timm forward_head/global_pool path
def _fsfm_extract_submit_hotfix(self, x):
    if hasattr(self.backbone, "forward_features"):
        out = self.backbone.forward_features(x)
    else:
        out = self.backbone(x)
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state[:, 0]
    if isinstance(out, (tuple, list)):
        return out[0][:, 0]
    return out
FSFMModel._extract = _fsfm_extract_submit_hotfix
# DINOv3 ViT-H inference hotfix: current tokens/RoPE API
def _dinov3_vith_extract_submit_hotfix(self, x):
    if hasattr(self.backbone, "prepare_tokens_with_masks"):
        tok, hw = self.backbone.prepare_tokens_with_masks(x, masks=None)
    else:
        tok = self.backbone.prepare_tokens(x)
        hw = None
    rope = None
    if hw is not None and getattr(self.backbone, "rope_embed", None) is not None:
        H, W = hw
        rope = self.backbone.rope_embed(H=H, W=W)
    cls_list = []
    for i, block in enumerate(self.backbone.blocks):
        tok = block(tok, rope)
        if i in self.INTERMEDIATE:
            cls_list.append(tok[:, 0])
    tok = self.backbone.norm(tok)
    cls_list.append(tok[:, 0])
    return torch.cat(cls_list, dim=-1)
DINOv3ViTHModel._extract = _dinov3_vith_extract_submit_hotfix
def submission_only(
    checkpoints,
    run_name="final_submission_only",
    n_tta=3,
):
    test_orig = build_test_samples(TEST_DIR)
    if not test_orig:
        print(f"[SKIP] No test images found under {TEST_DIR}")
        return None
    orig_stems = [p.stem for p, _ in test_orig]
    all_probs = []
    for name, (ckpt_path, cfg) in checkpoints.items():
        if not ckpt_path.exists():
            print(f"[SKIP] {name}: {ckpt_path} not found")
            continue
        print(f"\n── {name} ──")
        cache = CACHE_DIR / str(cfg.resolution)
        path_map = preprocess_all(test_orig, cache, cfg.resolution, max_workers=NUM_WORKERS)
        test_cached = [(path_map[str(p)], l) for p, l in test_orig]
        model = load_for_inference(ckpt_path, cfg)
        probs, _ = infer_model(model, test_cached, orig_stems, cfg, n_tta=n_tta)
        all_probs.append(probs)
        np.save(CKPT_DIR / f"probs_{name}.npy", probs)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if not all_probs:
        print("ERROR: no checkpoints available")
        return None
    ensemble = np.mean(all_probs, axis=0)
    sub_path = CKPT_DIR / f"submission_{run_name}.csv"
    write_submission(orig_stems, ensemble, sub_path)
    print(f"\nSubmission written to: {sub_path}")
    return sub_path
checkpoints = {
    "dinov3_cnx":  (CKPT_DIR / "dinov3_cnx.pt",  DINOV3_CNX_CFG),
    "fsfm":        (CKPT_DIR / "fsfm.pt",        FSFM_CFG),
    "dinov3_vith": (CKPT_DIR / "dinov3_vith.pt", DINOV3_VITH_CFG),
}
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
try:
    submission_path = submission_only(checkpoints, run_name="final_3model_noeval", n_tta=3)
except Exception as e:
    msg = str(e).strip()
    print(f"[ERROR] submission: {type(e).__name__}: {msg if msg else repr(e)}")
    traceback.print_exc(limit=10)


── dinov3_cnx ──
Cache: all 404 present at /dev/shm/fas_cache/384
Building DINOv3 ConvNeXt-Large ...
  Loaded via torch.hub: dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth
  Stem expanded to 6-channel (RGB weights replicated for FFT channels)
  ConvNeXt-L: 196,273,351 total | 196,273,351 trainable (100.00%)
  dinov3_cnx.pt: all 352 keys loaded ✓
  epoch=11 f1=0.9701767339415298
  Loading 404 test images ...


  0%|          | 0/404 [00:00<?, ?it/s]

  0%|          | 0/404 [00:00<?, ?it/s]

  Precomputing FFT channels ...


  FFT full:   0%|          | 0/404 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Done: 404/404
Building DINOv3 ViT-H+ ...
  Loaded via torch.hub: dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth
  LoRA injected into 32 linear layers (r=8, alpha=16)
  Trainable: 1,310,720/841,944,320 (0.16%)
  ViT-H+: 842,028,807 total | 1,395,207 trainable (0.17%)
  dinov3_vith.pt: all 624 keys loaded ✓
  epoch=15 f1=0.9787068162047813
  Loading 404 test images ...


  0%|          | 0/404 [00:00<?, ?it/s]

  0%|          | 0/404 [00:00<?, ?it/s]

  TTA 1/3 done
  TTA 2/3 done
  TTA 3/3 done

[Diagnostic] 5 binary/6-class disagreements:
  test_003                       6cls=fake_mask           (0.57) bin=realperson
  test_007                       6cls=fake_mask           (0.55) bin=realperson
  test_018                       6cls=fake_mask           (0.74) bin=realperson
  test_065                       6cls=fake_mask           (0.48) bin=realperson
  test_120                       6cls=fake_mask           (0.69) bin=realperson
Submission: /content/drive/MyDrive/faris/checkpoint/submission_final_3model_noeval.csv  (404 rows)
  fake_mannequin      : 52
  fake_mask           : 72
  fake_printed        : 58
  fake_screen         : 71
  fake_unknown        : 48
  realperson          : 103

Submission written to: /content/drive/MyDrive/faris/checkpoint/submission_final_3model_noeval.csv


## Penjelasan Cell Submission Saja (Tanpa Evaluasi)

Cell ini dibuat untuk kondisi finalisasi: memuat checkpoint yang tersedia, melakukan ensemble, lalu menulis CSV submission tanpa menghitung skor terhadap label tersembunyi. Cocok dipakai saat Anda ingin ekspor cepat hasil akhir.

# Penyempurnaan Submission dengan Meta-Layer

Bagian ini menerapkan post-processing berbasis confidence pada probabilitas model yang sudah diekspor.
Proses ini tidak membaca label target tersembunyi dan tidak melakukan scoring terhadap ground truth privat.
Tujuannya adalah mempertahankan beberapa varian submission dengan strategi fusi yang bisa direproduksi.

## Input yang Dibutuhkan

- probs_dinov3_vith.npy
- probs_dinov3_cnx.npy
- probs_fsfm.npy untuk varian yang memakai recovery tambahan
- dinov3_vith_test_submission.csv
- submission_rotated_ml.csv
- test_probs_rotated_ml.npy

## Varian File Output

- submission_algorithmic_meta.csv: koreksi batas keputusan yang lebih konservatif dengan eskalasi screen terjaga.
- harusnyabenerv2.csv: recovery kelas printed yang lebih agresif saat confidence realperson dari DINO melemah dan sinyal model lain mendukung.
- nt.csv: kombinasi recovery printed ditambah recovery screen ke realperson saat confidence screen DINO hanya moderat dan disagreement antar-model tinggi.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Callable, Dict, Tuple

META_FILES = {
    "ids": "dinov3_vith_test_submission.csv",
    "p_vith": "probs_dinov3_vith.npy",
    "p_cnx": "probs_dinov3_cnx.npy",
    "p_fsfm": "probs_fsfm.npy",
    "ids_ml": "submission_rotated_ml.csv",
    "p_ml": "test_probs_rotated_ml.npy",
}


def resolve_meta_root() -> Path:
    candidates = [
        PROJECT_ROOT,
        PROJECT_ROOT.parent,
        CKPT_DIR,
        FARIS,
    ]
    required = [META_FILES["ids"], META_FILES["p_vith"], META_FILES["p_cnx"], META_FILES["ids_ml"], META_FILES["p_ml"]]
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists() and all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(
        "Meta-layer artifacts were not found. Expected at least: "
        + ", ".join(required)
    )


META_ROOT = resolve_meta_root()
print(f"META_ROOT: {META_ROOT}")


def _stem_series(series: pd.Series) -> np.ndarray:
    return series.astype(str).map(lambda x: Path(x).stem).to_numpy()


def load_meta_inputs() -> Dict[str, np.ndarray]:
    ids_base = _stem_series(pd.read_csv(META_ROOT / META_FILES["ids"])["id"])
    ids_ml = _stem_series(pd.read_csv(META_ROOT / META_FILES["ids_ml"])["id"])
    p_vith = np.load(META_ROOT / META_FILES["p_vith"])
    p_cnx = np.load(META_ROOT / META_FILES["p_cnx"])
    p_ml_raw = np.load(META_ROOT / META_FILES["p_ml"])
    p_fsfm = np.load(META_ROOT / META_FILES["p_fsfm"]) if (META_ROOT / META_FILES["p_fsfm"]).exists() else None

    if len(ids_base) != len(p_vith) or len(ids_base) != len(p_cnx):
        raise ValueError("Base submission IDs do not align with DINO probability arrays.")
    if p_fsfm is not None and len(ids_base) != len(p_fsfm):
        raise ValueError("FSFM probability array length does not match base submission IDs.")

    ml_lookup = {sid: i for i, sid in enumerate(ids_ml)}
    missing = [sid for sid in ids_base if sid not in ml_lookup]
    if missing:
        raise KeyError(f"Missing rotated-ML probabilities for {len(missing)} ids, e.g. {missing[:5]}")

    p_ml = np.vstack([p_ml_raw[ml_lookup[sid]] for sid in ids_base])
    return {
        "ids": ids_base,
        "P_v": p_vith,
        "P_c": p_cnx,
        "P_f": p_fsfm,
        "P_m": p_ml,
    }


def pipeline_conservative(P_v: np.ndarray, P_c: np.ndarray, P_m: np.ndarray, P_f: np.ndarray = None) -> np.ndarray:
    preds = P_v.argmax(axis=1)
    preds[(P_v[:, 2] > 0.18) & (P_m[:, 1] > 0.80) & (preds == 1)] = 2
    preds[(P_v[:, 5] > 0.30) & (P_m[:, 1] > 0.80) & (preds == 1)] = 5
    preds[(P_c[:, 3] > 0.40) & (P_v[:, 2] < 0.30) & (preds == 2)] = 3
    preds[(P_v[:, 5] > 0.50) & (P_c[:, 2] > 0.20) & (P_m[:, 5] < 0.60) & (preds == 5)] = 2
    return preds


def pipeline_printed_recovery(P_v: np.ndarray, P_c: np.ndarray, P_m: np.ndarray, P_f: np.ndarray = None) -> np.ndarray:
    preds = P_v.argmax(axis=1)
    preds[(P_v[:, 2] > 0.18) & (P_m[:, 1] > 0.80) & (preds == 1)] = 2
    preds[(P_v[:, 5] > 0.30) & (P_m[:, 1] > 0.80) & (preds == 1)] = 5
    preds[(P_c[:, 3] > 0.40) & (preds == 2)] = 3
    preds[(P_v[:, 5] > 0.50) & (P_v[:, 5] < 0.65) & (P_c[:, 2] > 0.20) & (preds == 5)] = 2
    preds[(P_v[:, 5] > 0.50) & (P_c[:, 2] > 0.20) & (P_m[:, 5] < 0.60) & (preds == 5)] = 2
    preds[(P_v[:, 5] > 0.35) & (P_v[:, 5] < 0.65) & (P_v[:, 2] > 0.25) & (P_m[:, 5] < 0.90) & (preds == 5)] = 2
    return preds


def pipeline_screen_recovery(P_v: np.ndarray, P_c: np.ndarray, P_m: np.ndarray, P_f: np.ndarray = None) -> np.ndarray:
    preds = P_v.argmax(axis=1)
    preds[(P_v[:, 2] > 0.18) & (P_m[:, 1] > 0.80) & (preds == 1)] = 2
    preds[(P_v[:, 5] > 0.30) & (P_m[:, 1] > 0.80) & (preds == 1)] = 5
    preds[(P_c[:, 3] > 0.40) & (preds == 2)] = 3
    preds[(P_v[:, 5] > 0.50) & (P_c[:, 2] > 0.20) & (P_m[:, 5] < 0.60) & (preds == 5)] = 2
    mask_scr_rp = (
        (P_v[:, 3] > 0.30)
        & (P_v[:, 3] < 0.45)
        & (P_c[:, 3] < 0.10)
        & (preds == 3)
    )
    preds[mask_scr_rp] = 5
    return preds


def export_meta_variant(filename: str, rule_fn: Callable[..., np.ndarray]) -> Path:
    meta = load_meta_inputs()
    preds = rule_fn(meta["P_v"], meta["P_c"], meta["P_m"], meta["P_f"])
    out = pd.DataFrame({
        "id": meta["ids"],
        "label": [IDX_TO_CLASS[int(p)] for p in preds],
    })
    out_path = PROJECT_ROOT / filename
    out.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(out)} rows)")
    print(out["label"].value_counts().sort_index().to_string())
    return out_path


## Penjelasan Cell Fungsi Meta-Layer

Cell ini memuat utilitas meta-layer: resolusi lokasi file artefak, penyelarasan ID antar sumber probabilitas, definisi pipeline aturan confidence, dan fungsi ekspor setiap varian submission. Pastikan semua file input tersedia sebelum menjalankan cell ini.

## Ekspor Varian Meta-Layer

Jalankan cell berikut setelah file probabilitas dasar dari model tersedia.
Setiap file output menggunakan tensor probabilitas yang sama, namun menerapkan profil threshold yang berbeda agar tersedia beberapa opsi submission.

In [ ]:
META_PIPELINES = [
    ("submission_algorithmic_meta.csv", pipeline_conservative),
    ("harusnyabenerv2.csv", pipeline_printed_recovery),
    ("nt.csv", pipeline_screen_recovery),
]

meta_outputs = {}
for filename, rule_fn in META_PIPELINES:
    print("\n" + "=" * 72)
    print(f"Exporting {filename}")
    print("=" * 72)
    meta_outputs[filename] = export_meta_variant(filename, rule_fn)

meta_outputs
